# POLARIS — Training Pipeline

**Priority-Optimized Location and Risk-Adaptive Infrastructure Siting**
University of the Immaculate Conception — College of Computer Studies, Davao City
Study area: Talomo District, Davao City (14 barangays)

---

## What this notebook does

It turns raw spatial data into one file, `candidate_sites.json`, which the deployed
web application reads and displays. It computes nothing at run time in the browser.

The notebook runs top to bottom in six stages. Every stage ends with printed output,
a saved table, or a saved figure, so the run can be inspected without re-executing it.

| Stage | Cells | Produces |
|---|---|---|
| 0 — Setup | 1–2 | Configuration; the single source of truth for weights, thresholds and gates |
| 1 — Formula library | 3 | Every manuscript equation, implemented once as a named function |
| 2 — Spatial data | 4–7 | Candidate grid with all six features, hazard classes, barangay, shoreline distance |
| 3 — Scoring | 8 | Baseline MCDA composite score (the supervised training target) |
| 4 — Model | 9–11 | Trained Random Forest, held-out metrics, importances, linear-overlay comparison |
| 5 — Outputs | 12–14 | Classification, gates, recommendation levels, interpretation text, `candidate_sites.json` |
| 6 — Validation & reporting | 15–19 | Ablation, population proxy check, external checks, figures, synchronisation report |

## Manuscript ↔ code correspondence

Stage 1 declares a formula library. Every equation printed in the manuscript's
Mathematical Formalization section and Appendix B exists there exactly once, under an
ID (`F1` … `F22`). No later cell re-derives any of them. Cell 3 prints a register
table mapping each ID to its manuscript section and writes it to
`formula_register.csv`, so a panel member can hold the manuscript beside the run log
and check the correspondence line by line.

Two equations that the manuscript states but a library such as scikit-learn executes
internally — the ensemble mean (`F6`) and the variance-minimising split rule (`F7`) —
are **verified rather than reimplemented**: the notebook recomputes them from the
fitted trees and asserts agreement with the library output. That is stronger evidence
than a hand-written copy of the same equation.

## What must be supplied before running

1. Google Drive mounted, with the project folder laid out as in Cell 2.
2. The OpenStreetMap PBF extract for the Philippines.
3. The three Project NOAH shapefiles for Davao del Sur.
4. The PSA 2024 POPCEN barangay workbook, for Cell 16 only. Every other cell runs
   without it; Cell 16 reports `PENDING` and substitutes nothing.

## Reproducibility

`random_state = 42` throughout. Metric computation is in EPSG:32651 (WGS 84 / UTM 51N);
geographic coordinates are used only for display and export. The OSM snapshot date is
recorded in the export metadata, so any published figure is traceable to one state of a
continuously edited source.

## STAGE 0 — SETUP

### Cell 1 — Environment

Installs and imports everything the notebook uses, then prints library versions.
Nothing here is specific to POLARIS; the cell exists so that a run log records the
exact environment that produced the numbers.

In [ ]:
# =============================================================================
# CELL 1 — Environment
# Installs dependencies and imports every library used downstream. Prints versions
# so the run log identifies the environment that produced the results.
# =============================================================================
!pip install --upgrade pip -q
!pip install geopandas folium shapely scikit-learn scipy pyrosm osmnx openpyxl xlsxwriter -q

import os, re, glob, json, time, unicodedata, warnings, difflib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt

from shapely.geometry import Point
from shapely.ops import unary_union

from scipy.spatial import cKDTree
from scipy.stats import spearmanr, pearsonr

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

print('NumPy      :', np.__version__)
print('Pandas     :', pd.__version__)
print('GeoPandas  :', gpd.__version__)
import sklearn; print('scikit-learn:', sklearn.__version__)

### Cell 2 — Configuration: the single source of truth

Every value that the manuscript also prints is declared here and nowhere else: the
six criteria weights, the three classification thresholds, the shoreline setback
width, the grid spacing, the building-density radius, and the random seed.

`MANUSCRIPT_WEIGHTS` is a transcription of the values currently printed in the
manuscript. `verify_weights()` asserts the two dictionaries are identical before any
score is computed, so a divergence stops the run instead of silently producing a
dataset that the manuscript misdescribes. The same values are written to
`weights_manifest.json` and, later, into the export metadata.

Change a weight in one place only — here — and re-run.

In [ ]:
# =============================================================================
# CELL 2 — Configuration (single source of truth)
# Declares paths, the study area, the candidate grid, the six criteria weights,
# the classification thresholds, and the two hard gates. verify_weights() asserts
# that the pipeline values equal the values transcribed from the manuscript, and
# raises before any scoring happens if they diverge.
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')

PROJECT_FOLDER = 'polaris_Thesis'
DRIVE_BASE = f'/content/drive/MyDrive/{PROJECT_FOLDER}/Model'
DATA_DIR   = os.path.join(DRIVE_BASE, 'data')
OUTPUT_DIR = os.path.join(DRIVE_BASE, 'outputs_district')
MODEL_DIR  = os.path.join(OUTPUT_DIR, 'model')
WEBAPP_DIR = os.path.join(OUTPUT_DIR, 'webapp')
for d in (OUTPUT_DIR, MODEL_DIR, WEBAPP_DIR):
    os.makedirs(d, exist_ok=True)

OSM_LOCAL_DIR = '/content/drive/MyDrive/New Philippine OSM files'

# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_STATE   = 42
GEOGRAPHIC_CRS = 'EPSG:4326'
PROJECTED_CRS  = 'EPSG:32651'        # WGS 84 / UTM 51N — metres over Davao
np.random.seed(RANDOM_STATE)

# ── Study area ───────────────────────────────────────────────────────────────
STUDY_AREA = 'Talomo District'
TALOMO_BARANGAYS = [
    'Bago Aplaya', 'Bago Gallera', 'Baliok', 'Bucana',
    'Catalunan Grande', 'Catalunan Pequeño', 'Dumoy', 'Langub',
    'Ma-a', 'Magtuod', 'Matina Aplaya', 'Matina Crossing',
    'Matina Pangi', 'Talomo',
]
EXPECTED_BARANGAY_COUNT = 14
DAVAO_COARSE_BBOX = (125.30, 6.90, 125.80, 7.40)
BRGY_ADMIN_LEVEL  = '10'

# ── Candidate grid and feature windows ───────────────────────────────────────
GRID_SPACING_M           = 250   # metric spacing; matches the 250 m proximity warning
RADIUS_M                 = 300   # building-density search radius
ROAD_WINDOW_M            = 2000
FACILITY_SEARCH_BUFFER_M = 5000
COAST_WINDOW_M           = 5000
HEALTHCARE_AMENITIES = ['hospital', 'clinic', 'health_post', 'doctors']

# ── Hard eligibility gates ───────────────────────────────────────────────────
SHORELINE_BUFFER_M  = 50
EXCLUDE_HIGH_HAZARD = True

# ── Classification and recommendation ────────────────────────────────────────
TAU1, TAU2, TAU3      = 0.25, 0.40, 0.60
TOP_N_RECOMMENDATIONS = 3

# ── Weights ──────────────────────────────────────────────────────────────────
SERVICE_FEATURES = ['building_density', 'road_accessibility', 'facility_distance']
HAZARD_FEATURES  = ['flood_susceptibility', 'landslide_susceptibility',
                    'storm_surge_susceptibility']
ALL_FEATURES = SERVICE_FEATURES + HAZARD_FEATURES

SERVICE_WEIGHTS = {'building_density': 0.40,       # w1
                   'road_accessibility': 0.25,     # w2
                   'facility_distance': 0.15}      # w3
HAZARD_PENALTY_WEIGHTS = {'flood_susceptibility': 0.08,        # w4
                          'landslide_susceptibility': 0.07,    # w5
                          'storm_surge_susceptibility': 0.05}  # w6
WEIGHTS = {**SERVICE_WEIGHTS, **HAZARD_PENALTY_WEIGHTS}

# Transcribed from the manuscript. If these differ from WEIGHTS, the cell fails.
MANUSCRIPT_WEIGHTS = {'building_density': 0.40, 'road_accessibility': 0.25,
                      'facility_distance': 0.15, 'flood_susceptibility': 0.08,
                      'landslide_susceptibility': 0.07,
                      'storm_surge_susceptibility': 0.05}


def verify_weights():
    """Assert pipeline weights == manuscript weights, and that the groups sum
    to 0.80 (service) and 0.20 (hazard penalty)."""
    assert abs(sum(SERVICE_WEIGHTS.values()) - 0.80) < 1e-9, 'service weights must sum to 0.80'
    assert abs(sum(HAZARD_PENALTY_WEIGHTS.values()) - 0.20) < 1e-9, 'hazard weights must sum to 0.20'
    bad = {k: (WEIGHTS[k], MANUSCRIPT_WEIGHTS.get(k))
           for k in WEIGHTS if abs(WEIGHTS[k] - MANUSCRIPT_WEIGHTS.get(k, -1)) > 1e-9}
    if bad:
        for k, (a, b) in bad.items():
            print(f'   MISMATCH {k}: pipeline={a} manuscript={b}')
        raise AssertionError('Pipeline weights differ from MANUSCRIPT_WEIGHTS. '
                             'Reconcile one of the two before scoring.')
    print('Weights verified — service Σ=0.80, hazard Σ=0.20, pipeline == manuscript.')


verify_weights()

with open(os.path.join(OUTPUT_DIR, 'weights_manifest.json'), 'w') as f:
    json.dump({'service_weights': SERVICE_WEIGHTS,
               'hazard_penalty_weights': HAZARD_PENALTY_WEIGHTS,
               'thresholds': {'tau1': TAU1, 'tau2': TAU2, 'tau3': TAU3},
               'shoreline_buffer_m': SHORELINE_BUFFER_M,
               'grid_spacing_m': GRID_SPACING_M,
               'building_density_radius_m': RADIUS_M,
               'random_state': RANDOM_STATE,
               'generated_utc': datetime.now(timezone.utc).isoformat()}, f, indent=2)
print('weights_manifest.json written.')

# ── Hazard layers ────────────────────────────────────────────────────────────
NOAH_LAYERS = {
    'flood_susceptibility': {
        'shp': os.path.join(DATA_DIR, 'Flood_DavaoDelSur', 'DavaoDelSur_Flood_5year.shp'),
        'class_field': 'Var'},
    'landslide_susceptibility': {
        'shp': os.path.join(DATA_DIR, 'LandSlide_DavaoDelSur', 'DavaoDelSur_LandslideHazards.shp'),
        'class_field': 'LH'},
    'storm_surge_susceptibility': {
        'shp': os.path.join(DATA_DIR, 'StormSurge_DavaoDelSur', 'DavaoDelSur_StormSurge_SSA4.shp'),
        'class_field': 'HAZ'},
}
HAZARD_LABELS = {0: 'No Hazard', 1: 'Low', 2: 'Moderate', 3: 'High'}

# ── Class and level vocabulary (must match the web application exactly) ──────
CLASS_NOT, CLASS_LOW = 'Not Suitable', 'Low Suitability'
CLASS_MOD, CLASS_HIGH = 'Moderately Suitable', 'Highly Suitable'
SUITABILITY_CLASSES = [CLASS_HIGH, CLASS_MOD, CLASS_LOW, CLASS_NOT]
CLASS_LEVEL_RANGE = {CLASS_NOT: (0, 1), CLASS_LOW: (1, 2),
                     CLASS_MOD: (2, 3), CLASS_HIGH: (3, 4)}
LEVEL_LABELS = {0: 'Not Recommended', 1: 'Conditionally Suitable',
                2: 'Suitable with Reservations', 3: 'Recommended',
                4: 'Highly Recommended'}

print('\n── Hazard files')
for k, v in NOAH_LAYERS.items():
    print(f'   {k:28s}: {"OK     " if os.path.exists(v["shp"]) else "MISSING"} {v["shp"]}')
print(f'\n── OSM extract folder: {OSM_LOCAL_DIR}')
print('   ' + ('OK' if os.path.isdir(OSM_LOCAL_DIR) else 'MISSING'))

## STAGE 1 — FORMULA LIBRARY

### Cell 3 — Every manuscript equation, implemented once

This is the cell that makes the manuscript and the pipeline the same object.

Each formula in the manuscript's Mathematical Formalization section, plus the three
supplementary measures in Appendix B and the coverage metric in Tier 3, appears here
as one function carrying an ID. Downstream cells call these functions; none of them
re-implements the arithmetic. If a formula changes in the manuscript, it changes here,
and the change propagates.

Two equations are verified rather than reimplemented, because scikit-learn executes
them internally and a hand-written copy would prove nothing about what actually ran:

- **F6**, the ensemble mean, is checked by averaging the individual tree predictions
  and comparing against `model.predict`.
- **F7**, the variance-minimising split rule, is checked by recomputing the weighted
  child impurity at the root of the first tree from the fitted tree structure.

Both checks execute in Cell 10 and print their residuals.

The cell prints a register table and writes `formula_register.csv`. That table is
the artefact to bring to a defence: it lists every equation, its ID, the function that
implements it, and the manuscript section it appears in.

In [ ]:
# =============================================================================
# CELL 3 — Manuscript formula library
# One implementation per manuscript equation, each tagged with a formula ID.
# No later cell re-implements any of these. Prints the formula register and
# writes formula_register.csv.
# =============================================================================
FORMULA_REGISTER = []


def _reg(fid, name, expr, section, impl):
    FORMULA_REGISTER.append({'ID': fid, 'Formula': name, 'Expression': expr,
                             'Manuscript section': section, 'Implemented by': impl})


# ── F1 — Min–Max normalisation ───────────────────────────────────────────────
def f1_minmax(x):
    """x' = (x - x_min) / (x_max - x_min); zero-variance -> 0."""
    x = np.asarray(x, dtype=float)
    lo, hi = np.nanmin(x), np.nanmax(x)
    if not np.isfinite(hi - lo) or (hi - lo) == 0:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


_reg('F1', 'Min–Max normalisation', "x'ᵢ = (xᵢ − x_min)/(x_max − x_min)",
     'Min–Max Normalization', 'f1_minmax')


# ── F2 — Road accessibility metric ───────────────────────────────────────────
def f2_road_accessibility(d_metres):
    """RAᵢ = 1 / (1 + dᵢ), d in metres. Bounded (0, 1]. Higher = closer to a road."""
    return 1.0 / (1.0 + np.asarray(d_metres, dtype=float))


_reg('F2', 'Road accessibility metric', 'RAᵢ = 1 / (1 + dᵢ)',
     'Road Accessibility Metric', 'f2_road_accessibility')


# ── F3 — Hazard-integrated baseline composite score ──────────────────────────
def f3_composite_hazard_integrated(frame):
    """Sᴮᵢ = w₁·BD' + w₂·RA' + w₃·FD' − w₄·FL' − w₅·LS' − w₆·SS'"""
    s = sum(WEIGHTS[f] * frame[f] for f in SERVICE_FEATURES)
    s = s - sum(WEIGHTS[f] * frame[f] for f in HAZARD_FEATURES)
    return np.asarray(s, dtype=float)


_reg('F3', 'Hazard-integrated composite',
     'Sᴮᵢ = w₁BD′ + w₂RA′ + w₃FD′ − w₄FL′ − w₅LS′ − w₆SS′',
     'Baseline Composite Suitability Score', 'f3_composite_hazard_integrated')


# ── F4 — Service-only composite (internal ablation counterfactual) ───────────
def f4_composite_service_only(frame):
    """Sᴬᵢ = w₁·BD' + w₂·RA' + w₃·FD'. Never trained, never exported."""
    return np.asarray(sum(WEIGHTS[f] * frame[f] for f in SERVICE_FEATURES), dtype=float)


_reg('F4', 'Service-only composite', 'Sᴬᵢ = w₁BD′ + w₂RA′ + w₃FD′',
     'Baseline Composite Suitability Score', 'f4_composite_service_only')


# ── F5 — Baseline score rescaling ────────────────────────────────────────────
def f5_rescale(raw, bounds_mask=None):
    """S'ᵢ = (Sᵢ − S_min)/(S_max − S_min), clipped to [0, 1].

    bounds_mask selects the rows that define S_min and S_max. Sites inside the
    shoreline setback are excluded from the bounds and clipped afterwards, so the
    eligible population scores identically to a delete-then-score pipeline.
    """
    raw = np.asarray(raw, dtype=float)
    src = raw if bounds_mask is None else raw[bounds_mask]
    lo, hi = float(np.nanmin(src)), float(np.nanmax(src))
    if hi - lo == 0:
        return np.zeros_like(raw), lo, hi
    return np.clip((raw - lo) / (hi - lo), 0.0, 1.0), lo, hi


_reg('F5', 'Baseline score rescaling', "S′ᵢ = (Sᵢ − S_min)/(S_max − S_min)",
     'Baseline Score Rescaling', 'f5_rescale')


# ── F6 — Random Forest ensemble prediction (VERIFIED, not reimplemented) ─────
def f6_verify_ensemble_mean(fitted_model, X, tol=1e-9):
    """ŷ(x) = (1/B) Σⱼ Tⱼ(x).

    Averages the individual fitted trees and compares against model.predict.
    Returns the maximum absolute residual; asserts it is within tolerance.
    """
    manual = np.mean([t.predict(X) for t in fitted_model.estimators_], axis=0)
    resid = float(np.max(np.abs(manual - fitted_model.predict(X))))
    assert resid < tol, f'Ensemble mean does not reproduce model.predict (max |Δ| = {resid})'
    return resid


_reg('F6', 'Ensemble prediction', 'ŷ(x) = (1/B) Σⱼ Tⱼ(x)',
     'Random Forest Regression Model', 'f6_verify_ensemble_mean (verification)')


# ── F7 — Variance-minimising split criterion (VERIFIED) ──────────────────────
def f7_verify_split_criterion(fitted_model, tree_index=0):
    """(p*, s*) = arg min [ (n_l/n_ν)·Var(y_l) + (n_r/n_ν)·Var(y_r) ].

    For squared-error regression trees the node impurity IS the within-node
    variance, so the criterion is verified by recomputing the weighted child
    impurity at the root of one fitted tree and confirming that the realised
    split reduces the parent impurity.
    """
    assert fitted_model.criterion == 'squared_error', \
        f'criterion is {fitted_model.criterion}; the manuscript specifies within-node variance'
    t = fitted_model.estimators_[tree_index].tree_
    l, r = int(t.children_left[0]), int(t.children_right[0])
    n_p, n_l, n_r = (float(t.weighted_n_node_samples[0]),
                     float(t.weighted_n_node_samples[l]),
                     float(t.weighted_n_node_samples[r]))
    weighted_child = (n_l / n_p) * float(t.impurity[l]) + (n_r / n_p) * float(t.impurity[r])
    return {'tree': tree_index,
            'split_feature_index': int(t.feature[0]),
            'split_threshold': float(t.threshold[0]),
            'parent_variance': float(t.impurity[0]),
            'weighted_child_variance': weighted_child,
            'variance_reduction': float(t.impurity[0]) - weighted_child}


_reg('F7', 'Split criterion',
     '(p*, s*) = arg min [(nₗ/nᵥ)Var(yₗ) + (nᵣ/nᵥ)Var(yᵣ)]',
     'Random Forest Regression Model', 'f7_verify_split_criterion (verification)')


# ── F8 — Mean Decrease in Impurity ───────────────────────────────────────────
def f8_mdi(fitted_model, feature_names):
    """MDI(p) = (1/B) Σⱼ Σᵥ (nᵥ/N)·ΔVarᵥ, normalised to sum to 1."""
    return {f: float(v) for f, v in zip(feature_names, fitted_model.feature_importances_)}


_reg('F8', 'Mean Decrease in Impurity', 'MDI(p) = (1/B) Σⱼ Σᵥ (nᵥ/N)·ΔVarᵥ',
     'Feature Importance', 'f8_mdi')


# ── F9 — Permutation importance ──────────────────────────────────────────────
def f9_permutation_importance(fitted_model, X, y, feature_names, T=10):
    """PI(p) = (1/T) Σₜ (MSEₜ − MSE₀).

    scoring is fixed to neg_mean_squared_error so that the returned value is the
    mean INCREASE IN MSE, which is what the manuscript defines. The scikit-learn
    default scorer for a regressor is R², which would give the mean decrease in R²
    instead — a different quantity on a different scale.
    """
    res = permutation_importance(fitted_model, X, y, n_repeats=T,
                                 scoring='neg_mean_squared_error',
                                 random_state=RANDOM_STATE, n_jobs=-1)
    return ({f: float(v) for f, v in zip(feature_names, res.importances_mean)},
            {f: float(v) for f, v in zip(feature_names, res.importances_std)})


_reg('F9', 'Permutation importance', 'PI(p) = (1/T) Σₜ (MSEₜ − MSE₀)',
     'Feature Importance', 'f9_permutation_importance')


# ── F10–F12 — Model evaluation metrics ───────────────────────────────────────
def f10_mae(y, yhat):
    """MAE = (1/m) Σ |ŷᵢ − yᵢ|"""
    return float(np.mean(np.abs(np.asarray(yhat) - np.asarray(y))))


def f11_rmse(y, yhat):
    """RMSE = sqrt((1/m) Σ (ŷᵢ − yᵢ)²)"""
    return float(np.sqrt(np.mean((np.asarray(yhat) - np.asarray(y)) ** 2)))


def f12_r2(y, yhat):
    """R² = 1 − Σ(yᵢ − ŷᵢ)² / Σ(yᵢ − ȳ)²"""
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    return float(1.0 - ss_res / ss_tot)


_reg('F10', 'Mean Absolute Error', 'MAE = (1/m) Σ |ŷᵢ − yᵢ|', 'Model Evaluation Metrics', 'f10_mae')
_reg('F11', 'Root Mean Squared Error', 'RMSE = √[(1/m) Σ (ŷᵢ − yᵢ)²]', 'Model Evaluation Metrics', 'f11_rmse')
_reg('F12', 'Coefficient of Determination', 'R² = 1 − Σ(yᵢ−ŷᵢ)² / Σ(yᵢ−ȳ)²', 'Model Evaluation Metrics', 'f12_r2')
_reg('F13', 'Out-of-bag R²', 'R²_OOB over rows unseen by each bootstrap tree',
     'Model Evaluation Metrics', 'RandomForestRegressor(oob_score=True)')


# ── F14 — Suitability classification ─────────────────────────────────────────
def f14_classify(score):
    """Not Suitable < τ₁ ≤ Low Suitability < τ₂ ≤ Moderately Suitable < τ₃ ≤ Highly Suitable"""
    if score < TAU1:
        return CLASS_NOT
    if score < TAU2:
        return CLASS_LOW
    if score < TAU3:
        return CLASS_MOD
    return CLASS_HIGH


_reg('F14', 'Suitability classification', 'four classes at τ₁=0.25, τ₂=0.40, τ₃=0.60',
     'Suitability Classification', 'f14_classify')


# ── F15 — Hybrid recommendation level by intra-class percentile ──────────────
def f15_intraclass_percentile(scores):
    """Pᵢ = nᵤ / nᴄ — the proportion of same-class sites scoring STRICTLY LOWER.

    Implemented with rank(method='min'), which gives 1 + (number strictly lower);
    subtracting 1 and dividing by the class size yields exactly nᵤ/nᴄ. Note that
    rank(pct=True) would NOT: it averages ties and divides by n, producing a
    different quantity from the one the manuscript defines.
    """
    s = pd.Series(np.asarray(scores, dtype=float))
    return ((s.rank(method='min') - 1) / len(s)).values


_reg('F15', 'Intra-class percentile', 'Pᵢ = nᵤ / nᴄ; Pᵢ ≥ 0.50 → upper level',
     'Hybrid Recommendation Level', 'f15_intraclass_percentile')


# ── F16 — Spearman's rank correlation ────────────────────────────────────────
def f16_spearman(a, b):
    """ρ = 1 − 6Σdᵢ² / [n(n²−1)] under no ties.

    scipy.stats.spearmanr is used because the closed form above is exact only
    when no ranks are tied; scipy applies the tie-corrected definition, which is
    the correct generalisation. Both are reported so the difference is visible.
    """
    a, b = np.asarray(a, float), np.asarray(b, float)
    rho_scipy, p = spearmanr(a, b)
    n = len(a)
    d2 = float(np.sum((pd.Series(a).rank().values - pd.Series(b).rank().values) ** 2))
    rho_closed = 1.0 - (6.0 * d2) / (n * (n ** 2 - 1))
    return float(rho_scipy), float(p), float(rho_closed)


_reg('F16', "Spearman's rank correlation", 'ρ = 1 − 6Σdᵢ² / [n(n²−1)]',
     'Internal Hazard Sensitivity Validation Measures', 'f16_spearman')


# ── F17 — Classification Change Percentage ───────────────────────────────────
def f17_classification_change_pct(class_a, class_b):
    """C = (1/n) Σ 1[classᴬ(i) ≠ classᴮ(i)] × 100%"""
    a, b = np.asarray(class_a), np.asarray(class_b)
    changed = int(np.sum(a != b))
    return float(100.0 * changed / len(a)), changed


_reg('F17', 'Classification Change Percentage', 'C = (1/n) Σ 1[classᴬ ≠ classᴮ] × 100%',
     'Internal Hazard Sensitivity Validation Measures', 'f17_classification_change_pct')


# ── F18 — Mean Suitability Score Deviation (Appendix B) ──────────────────────
def f18_mean_deviation(s_service_raw, s_hazard_raw):
    """Δ̄ = (1/n) Σ (Sᴬᵢ − Sᴮᵢ), on RAW pre-rescaling scores."""
    return float(np.mean(np.asarray(s_service_raw) - np.asarray(s_hazard_raw)))


_reg('F18', 'Mean suitability score deviation', 'Δ̄ = (1/n) Σ (Sᴬᵢ − Sᴮᵢ)',
     'Appendix B', 'f18_mean_deviation')


# ── F19 — Top-N Jaccard similarity (Appendix B) ──────────────────────────────
def f19_jaccard_topn(s_service, s_hazard, N):
    """J(N) = |Aₙ ∩ Bₙ| / |Aₙ ∪ Bₙ| over the two top-N shortlists."""
    A = set(np.argsort(-np.asarray(s_service))[:N])
    B = set(np.argsort(-np.asarray(s_hazard))[:N])
    return float(len(A & B) / len(A | B)), len(A & B)


_reg('F19', 'Top-N Jaccard similarity', 'J(N) = |Aₙ ∩ Bₙ| / |Aₙ ∪ Bₙ|',
     'Appendix B', 'f19_jaccard_topn')


# ── F20 — Spatial difference mapping (Appendix B) ────────────────────────────
def f20_spatial_difference(s_service_raw, s_hazard_raw):
    """Δᵢ = Sᴬᵢ − Sᴮᵢ, per site, on RAW pre-rescaling scores."""
    return np.asarray(s_service_raw, float) - np.asarray(s_hazard_raw, float)


_reg('F20', 'Spatial difference mapping', 'Δᵢ = Sᴬᵢ − Sᴮᵢ', 'Appendix B',
     'f20_spatial_difference')


# ── F21 — Population coverage improvement (Tier 3) ───────────────────────────
def f21_delta_coverage(demand_xy, polaris_xy, current_xy, buffer_m):
    """ΔCoverage = %Coverage_POLARIS − %Coverage_Current, over demand points
    falling within a buffer of either configuration's facility set."""
    tree = cKDTree(demand_xy)
    cov = lambda pts: len(set().union(*[set(tree.query_ball_point(p, r=buffer_m))
                                        for p in pts])) if len(pts) else 0
    tot = len(demand_xy)
    p_pct, c_pct = 100.0 * cov(polaris_xy) / tot, 100.0 * cov(current_xy) / tot
    return {'polaris_pct': p_pct, 'current_pct': c_pct, 'delta_pct': p_pct - c_pct}


_reg('F21', 'Population coverage improvement',
     'ΔCoverage = %Cov_POLARIS − %Cov_Current', 'Tier 3 — External Validity',
     'f21_delta_coverage')


# ── F22 — Hard eligibility gates ─────────────────────────────────────────────
def f22_eligibility(shoreline_distance_m, hazard_class_matrix):
    """A site is ineligible if it lies within SHORELINE_BUFFER_M of the coastline,
    or carries ordinal class 3 (High) on any hazard layer. Ineligible sites are
    retained in the dataset with their true values, never deleted."""
    shore = np.asarray(shoreline_distance_m, float)
    shore_excl = np.nan_to_num(shore, nan=np.inf) < SHORELINE_BUFFER_M
    hz = np.asarray(hazard_class_matrix)
    high_excl = (hz == 3).any(axis=1) if EXCLUDE_HIGH_HAZARD else np.zeros(len(shore), bool)
    return shore_excl, high_excl, ~(shore_excl | high_excl)


_reg('F22', 'Hard eligibility gates',
     'ineligible if d_coast < 50 m OR any hazard class = 3', 'Hard Eligibility Gates',
     'f22_eligibility')

def f5_shared_rescale(raw_service, raw_hazard, bounds_mask=None):
    """PATCH (post-first-run finding). F5 rescales S^A and S^B independently,
    each against its own min/max. That is correct for F16 (rank-based, scale-
    invariant) but WRONG for F17: a zero-hazard site can cross a class threshold
    under S^A vs S^B purely because the two configurations occupy different
    numeric ranges, with no hazard penalty involved at all. In the first run
    this alone produced C = 80.09%, which is not a hazard-attributable figure.

    Fix: rescale BOTH configurations against the hazard-integrated bounds
    (S_min, S_max from raw_hazard, over the same retained-site mask used
    elsewhere). This isolates the hazard subtraction as the only difference
    between the two classified outputs. Used ONLY for F17 in Cell 15 — F16
    and the magnitude measures (F18, F20) are unaffected by this and keep
    their existing definitions.
    """
    raw_hazard = np.asarray(raw_hazard, dtype=float)
    src = raw_hazard if bounds_mask is None else raw_hazard[bounds_mask]
    lo, hi = float(np.nanmin(src)), float(np.nanmax(src))
    if hi - lo == 0:
        z = np.zeros_like(raw_hazard)
        return z, z.copy(), lo, hi
    clip = lambda a: np.clip((np.asarray(a, dtype=float) - lo) / (hi - lo), 0.0, 1.0)
    return clip(raw_service), clip(raw_hazard), lo, hi


_reg('F5-shared', 'Common-scale rescale (ablation classification only)',
     "S'ᵢ = (Sᵢ − S_min^B)/(S_max^B − S_min^B), both configurations",
     'Baseline Score Rescaling (post-run correction)', 'f5_shared_rescale')

# ── Register ─────────────────────────────────────────────────────────────────
formula_register = pd.DataFrame(FORMULA_REGISTER)
formula_register.to_csv(os.path.join(OUTPUT_DIR, 'formula_register.csv'), index=False)
print(f'{len(formula_register)} manuscript formulas implemented in this cell.\n')
print(formula_register[['ID', 'Formula', 'Manuscript section']].to_string(index=False))
print('\nformula_register.csv written — bring this table to the defence alongside '
      'the Mathematical Formalization section.')

## STAGE 2 — SPATIAL DATA

### Cell 4 — Resolve the district from its 14 barangays

Finds the OSM extract, then builds the study-area polygon as the **union of the 14
named barangay polygons**, not by searching for a single polygon whose name contains
"Talomo". Davao City has a *barangay* called Talomo nested inside the *district* also
called Talomo; a name search returns the barangay and silently collapses the study
area to a fourteenth of its intended extent.

Name matching runs in three tiers — exact, containment, fuzzy — against an
accent-stripped, token-normalised form, so `Catalunan Pequeño`, `Catalunan Pequeno`
and `Barangay 76-A (Bucana)` all resolve. The same normaliser is reused for the census
join in Cell 16; using two different normalisers is what caused the census join to
return zero in an earlier run.

The cell **raises** if fewer than 14 barangays resolve. Do not work around this — a
partial match means every downstream figure describes a smaller area than the
manuscript claims.

In [ ]:
# =============================================================================
# CELL 4 — OSM source discovery, name normalisation, district geometry
# Locates the OSM extract, defines the shared barangay-name normaliser, resolves
# all 14 barangay polygons at admin_level=10, and unions them into the district
# polygon.
#
# [REV] Rewritten after a silent mis-resolution: canonical 'Talomo' matched BOTH
# 'Talomo Proper' (correct, Talomo District) and 'Talomo River' (a real barangay
# in CALINAN District, ~15 km north) via the containment rule, and the
# largest-area tie-break then selected the wrong one. The count still reached 14
# so the assert passed. Four guards now prevent this:
#   1. containment is restricted to numeric prefix codes ('76-A Bucana' only);
#   2. an explicit alias table resolves the known OSM label variants;
#   3. collisions are broken by proximity to the district cluster, not by area;
#   4. the resolved set must form ONE spatially contiguous block.
# =============================================================================
_DROP_TOKENS = ('barangay', 'brgy', 'bgy', 'poblacion', 'pob', 'proper', 'district')

# OSM labels that differ from the canonical names. Matched before anything else.
BARANGAY_ALIASES = {
    'Talomo'         : ['Talomo Proper'],
    'Bucana'         : ['76-A Bucana', 'Barangay 76-A (Bucana)'],
    'Matina Crossing': ['74-A Matina Crossing', 'Barangay 74-A (Matina Crossing)'],
    'Matina Aplaya'  : ['75-A Matina Aplaya', 'Barangay 75-A (Matina Aplaya)'],
}

# Known false positives. 'Talomo River' is a Calinan District barangay whose name
# contains 'Talomo'; it is not part of the study area.
NAME_BLOCKLIST = {'Talomo River'}
_BLOCKED_NORM = None  # populated below, after norm_name is defined

# Contiguity tolerance: OSM barangay polygons often have small gaps at shared
# edges, so adjacency is tested with this buffer rather than strict touching.
CONTIGUITY_TOL_M = 150

_PREFIX_CODE = re.compile(r'^\d+[a-z]?')


def norm_name(s):
    """Accent-stripped, lowercased, punctuation-free, administrative tokens removed.
    'Catalunan Pequeño' -> 'catalunanpequeno'; 'Talomo Proper' -> 'talomo'.
    Note 'river' is deliberately NOT a drop token: stripping it would collapse
    'Talomo River' onto 'talomo' and recreate the collision this cell fixes."""
    if s is None:
        return ''
    s = unicodedata.normalize('NFKD', str(s))
    s = ''.join(c for c in s if not unicodedata.combining(c)).lower()
    s = re.sub(r'[^a-z0-9]+', ' ', s)
    return ''.join(t for t in s.split() if t not in _DROP_TOKENS)


_BLOCKED_NORM = {norm_name(x) for x in NAME_BLOCKLIST}
_ALIAS_NORM = {canon: {norm_name(a) for a in alts}
               for canon, alts in BARANGAY_ALIASES.items()}


def name_matches(canonical, candidate, cutoff=0.90):
    """Returns (matched, tier, how). Lower tier = stronger evidence.

    tier 0  exact normalised equality, or a declared alias
    tier 1  numeric prefix code only ('76-A Bucana' -> 'bucana')
    tier 2  fuzzy, at a deliberately high cutoff

    Free containment is NOT permitted. 'Talomo River' normalises to
    'talomoriver', which contains 'talomo' but is a different barangay in a
    different district; admitting that match is what corrupted the study area.
    """
    a, b = norm_name(canonical), norm_name(candidate)
    if not a or not b:
        return False, 9, 'none'
    if b in _BLOCKED_NORM:
        return False, 9, 'blocked'
    if a == b:
        return True, 0, 'exact'
    if b in _ALIAS_NORM.get(canonical, ()):
        return True, 0, 'alias'
    if _PREFIX_CODE.sub('', b) == a and _PREFIX_CODE.match(b):
        return True, 1, 'prefix-code'
    if difflib.SequenceMatcher(None, a, b).ratio() >= cutoff:
        return True, 2, 'fuzzy'
    return False, 9, 'none'


# ── Locate the extract ───────────────────────────────────────────────────────
pbf, shp = [], []
if os.path.isdir(OSM_LOCAL_DIR):
    for root, _, files in os.walk(OSM_LOCAL_DIR):
        for fn in sorted(files):
            p = os.path.join(root, fn)
            if fn.lower().endswith('.pbf'):
                pbf.append(p)
            elif fn.lower().endswith('.shp'):
                shp.append(p)

if pbf:
    OSM_SOURCE_KIND, OSM_SOURCE_PATH = 'pbf', max(pbf, key=os.path.getsize)
elif any('gis_osm_' in os.path.basename(p) for p in shp):
    OSM_SOURCE_KIND = 'geofabrik_shp'
    OSM_SOURCE_PATH = os.path.dirname(
        [p for p in shp if 'gis_osm_' in os.path.basename(p)][0])
else:
    OSM_SOURCE_KIND, OSM_SOURCE_PATH = 'none', None
assert OSM_SOURCE_KIND != 'none', f'No usable OSM source in {OSM_LOCAL_DIR}'

m = re.search(r'(\d{6,8})', os.path.basename(OSM_SOURCE_PATH))
if m and len(m.group(1)) == 6:
    y, mo, d = m.group(1)[:2], m.group(1)[2:4], m.group(1)[4:6]
    OSM_SNAPSHOT_DATE = f'20{y}-{mo}-{d}'
else:
    OSM_SNAPSHOT_DATE = datetime.fromtimestamp(
        os.path.getmtime(OSM_SOURCE_PATH)).strftime('%Y-%m-%d')
print(f'OSM source    : {OSM_SOURCE_KIND} — {OSM_SOURCE_PATH}')
print(f'Snapshot date : {OSM_SNAPSHOT_DATE}   (override manually if wrong)')

# ── Administrative boundaries ────────────────────────────────────────────────
if OSM_SOURCE_KIND == 'pbf':
    from pyrosm import OSM
    boundaries = OSM(OSM_SOURCE_PATH, bounding_box=list(DAVAO_COARSE_BBOX)) \
        .get_boundaries(boundary_type='administrative')
else:
    cand = glob.glob(os.path.join(OSM_SOURCE_PATH, 'gis_osm_adminareas_a_free_*.shp'))
    boundaries = gpd.read_file(cand[0], bbox=DAVAO_COARSE_BBOX) if cand else None

assert boundaries is not None and len(boundaries), 'No administrative boundaries read.'
boundaries = boundaries[boundaries.geometry.notna()]
boundaries = boundaries[boundaries.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
boundaries = (boundaries.set_crs(GEOGRAPHIC_CRS) if boundaries.crs is None
              else boundaries.to_crs(GEOGRAPHIC_CRS))

lvl_col = next((c for c in ('admin_level', 'adminlevel', 'level') if c in boundaries.columns), None)
name_col = next((c for c in ('name', 'NAME') if c in boundaries.columns), 'name')
pool = (boundaries[boundaries[lvl_col].astype(str) == BRGY_ADMIN_LEVEL].copy()
        if lvl_col else boundaries.copy())
pool = pool.reset_index(drop=True)
_pool_proj = pool.to_crs(PROJECTED_CRS)
pool['_area'] = _pool_proj.geometry.area
pool['_cx'] = _pool_proj.geometry.centroid.x
pool['_cy'] = _pool_proj.geometry.centroid.y
print(f'admin_level={BRGY_ADMIN_LEVEL} candidate polygons: {len(pool)}')

# ── Collect every candidate hit per canonical name ───────────────────────────
hits_by_canon, unmatched = {}, []
for canon in TALOMO_BARANGAYS:
    hits = []
    for i, r in pool.iterrows():
        ok, tier, how = name_matches(canon, r[name_col])
        if ok:
            hits.append({'idx': i, 'osm_name': r[name_col], 'tier': tier,
                         'how': how, 'area_km2': r['_area'] / 1e6,
                         'cx': r['_cx'], 'cy': r['_cy']})
    if not hits:
        unmatched.append(canon)
        continue
    best_tier = min(h['tier'] for h in hits)
    hits_by_canon[canon] = [h for h in hits if h['tier'] == best_tier]

if unmatched:
    print(f'\nUNMATCHED: {unmatched}')
    print('Candidate admin_level=10 names containing any canonical fragment:')
    for canon in unmatched:
        frag = norm_name(canon)[:5]
        near = [r[name_col] for _, r in pool.iterrows() if frag in norm_name(r[name_col])]
        print(f'   {canon:20s} -> {near[:8]}')
    raise AssertionError(
        f'{len(unmatched)} of {EXPECTED_BARANGAY_COUNT} barangays did not match any '
        f'polygon. Add the correct OSM label to BARANGAY_ALIASES and re-run.')

_collisions = {c: h for c, h in hits_by_canon.items() if len(h) > 1}
if _collisions:
    print('\nName collisions at the strongest tier (resolved by proximity below):')
    for c, hs in _collisions.items():
        print(f'   {c:20s} -> ' + ', '.join(
            f'"{h["osm_name"]}" [{h["how"]}, {h["area_km2"]:.2f} km²]' for h in hs))

# ── Seed the cluster from unambiguous names, then resolve collisions ─────────
seed = {c: h[0] for c, h in hits_by_canon.items() if len(h) == 1}
assert seed, 'Every canonical name is ambiguous; cannot seed the district cluster.'
seed_cx = float(np.mean([h['cx'] for h in seed.values()]))
seed_cy = float(np.mean([h['cy'] for h in seed.values()]))
print(f'\nCluster seeded from {len(seed)} unambiguous barangay(s).')

chosen = dict(seed)
for canon, hs in _collisions.items():
    ranked = sorted(hs, key=lambda h: np.hypot(h['cx'] - seed_cx, h['cy'] - seed_cy))
    chosen[canon] = ranked[0]
    for h in ranked[1:]:
        d_km = np.hypot(h['cx'] - seed_cx, h['cy'] - seed_cy) / 1000
        print(f'   REJECTED "{h["osm_name"]}" for {canon}: {d_km:.1f} km from the '
              f'district cluster centre')

# ── Contiguity guard ─────────────────────────────────────────────────────────
# Talomo District's barangays form one connected block. A polygon that touches
# nothing else in the set is in the wrong district, whatever its name says.
def _largest_connected(names, geoms_proj, tol_m):
    n = len(names)
    buf = [g.buffer(tol_m) for g in geoms_proj]
    adj = {i: set() for i in range(n)}
    for i in range(n):
        for j in range(i + 1, n):
            if buf[i].intersects(buf[j]):
                adj[i].add(j)
                adj[j].add(i)
    seen, best = set(), []
    for i in range(n):
        if i in seen:
            continue
        stack, comp = [i], []
        seen.add(i)
        while stack:
            k = stack.pop()
            comp.append(k)
            for nb in adj[k]:
                if nb not in seen:
                    seen.add(nb)
                    stack.append(nb)
        if len(comp) > len(best):
            best = comp
    return set(best)


_names = list(chosen.keys())
_geoms_proj = [_pool_proj.loc[chosen[c]['idx'], 'geometry'] for c in _names]
_keep = _largest_connected(_names, _geoms_proj, CONTIGUITY_TOL_M)
_detached = [_names[i] for i in range(len(_names)) if i not in _keep]

if _detached:
    print('\nCONTIGUITY FAILURE — these do not touch the district block:')
    for c in _detached:
        print(f'   {c:20s} <- "{chosen[c]["osm_name"]}" '
              f'({chosen[c]["area_km2"]:.2f} km², {chosen[c]["how"]})')
    raise AssertionError(
        f'{len(_detached)} resolved polygon(s) are spatially detached from the '
        f'district. This is the Talomo River failure mode: a correctly-named '
        f'polygon from another district. Add the offending OSM name to '
        f'NAME_BLOCKLIST, or the correct one to BARANGAY_ALIASES, and re-run.')
print(f'\nContiguity check passed — all {len(_keep)} polygons form one block '
      f'(tolerance {CONTIGUITY_TOL_M} m).')

# ── Assemble ─────────────────────────────────────────────────────────────────
brgy_gdf = gpd.GeoDataFrame(
    [{'barangay': c, 'osm_name': chosen[c]['osm_name'],
      'match_type': chosen[c]['how'], 'area_km2': chosen[c]['area_km2'],
      'geometry': pool.loc[chosen[c]['idx'], 'geometry']}
     for c in TALOMO_BARANGAYS],
    crs=GEOGRAPHIC_CRS)

print(f'\nBarangays resolved: {len(brgy_gdf)} of {EXPECTED_BARANGAY_COUNT}')
print(brgy_gdf[['barangay', 'osm_name', 'match_type', 'area_km2']]
      .round(2).to_string(index=False))
assert len(brgy_gdf) == EXPECTED_BARANGAY_COUNT, (
    f'Only {len(brgy_gdf)} of {EXPECTED_BARANGAY_COUNT} barangays resolved.')
assert brgy_gdf['osm_name'].nunique() == EXPECTED_BARANGAY_COUNT, (
    'Two canonical names resolved to the SAME polygon: '
    f'{brgy_gdf[brgy_gdf.duplicated("osm_name", keep=False)]["osm_name"].tolist()}')

district_poly = unary_union(brgy_gdf.geometry.values)
district_proj = gpd.GeoSeries([district_poly], crs=GEOGRAPHIC_CRS) \
                   .to_crs(PROJECTED_CRS).iloc[0]
DISTRICT_AREA_KM2 = float(district_proj.area / 1e6)
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = district_poly.bounds

# No hole may remain: a hollow interior means a barangay is still missing.
_holes = sum(len(g.interiors) for g in
             (district_poly.geoms if district_poly.geom_type == 'MultiPolygon'
              else [district_poly]))
_parts = len(district_poly.geoms) if district_poly.geom_type == 'MultiPolygon' else 1
print(f'\n{STUDY_AREA}: {len(brgy_gdf)} barangays, {DISTRICT_AREA_KM2:.2f} km²')
print(f'   BBox LAT {LAT_MIN:.4f}–{LAT_MAX:.4f}  LON {LON_MIN:.4f}–{LON_MAX:.4f}')
print(f'   Geometry parts: {_parts}   interior holes: {_holes}')
if _holes:
    print('   WARNING: the district polygon has an interior hole. A hole means an '
          'enclosed barangay was not resolved — inspect before proceeding.')
if _parts > 1:
    print('   WARNING: the district polygon is in several pieces.')

# ── Visual verification ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8))
brgy_gdf.to_crs(PROJECTED_CRS).plot(ax=ax, facecolor='#dbeafe',
                                    edgecolor='#1e40af', linewidth=0.8)
for _, r in brgy_gdf.to_crs(PROJECTED_CRS).iterrows():
    c = r.geometry.centroid
    ax.annotate(r['barangay'], (c.x, c.y), fontsize=7, ha='center')
ax.set_title(f'{STUDY_AREA} — {len(brgy_gdf)} barangays, {DISTRICT_AREA_KM2:.1f} km²\n'
             f'no holes, one contiguous block')
ax.set_aspect('equal'); ax.set_axis_off()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'district_resolution.png'), dpi=150)
plt.show()

brgy_gdf[['barangay', 'osm_name', 'match_type', 'area_km2']] \
    .to_csv(os.path.join(OUTPUT_DIR, 'barangay_resolution.csv'), index=False)

### Cell 5 — Candidate site grid

Lays a regular point grid over the district in EPSG:32651, so spacing is a true
`GRID_SPACING_M` metres everywhere. A latitude/longitude mesh would not be: spacing in
metres would vary with latitude and the grid would not be uniform.

Points outside the district polygon are discarded. No jitter is applied — a
deterministic grid re-runs identically, a jittered one does not, and the jitter served
no analytical purpose.

The site count printed here is the figure `n` that the manuscript must carry
throughout. It will not equal 3,119; that number came from the superseded
single-barangay run and is not comparable.

In [ ]:
# =============================================================================
# CELL 5 — Candidate site grid
# Generates a metric-spaced point grid over the district polygon in EPSG:32651,
# clips it to the polygon, and assigns stable site IDs. N_SITES printed here is
# the value of n used throughout the manuscript.
# =============================================================================
minx, miny, maxx, maxy = district_proj.bounds
xs = np.arange(minx + GRID_SPACING_M / 2, maxx, GRID_SPACING_M)
ys = np.arange(miny + GRID_SPACING_M / 2, maxy, GRID_SPACING_M)
gx, gy = np.meshgrid(xs, ys)

grid = gpd.GeoDataFrame(geometry=[Point(x, y) for x, y in zip(gx.ravel(), gy.ravel())],
                        crs=PROJECTED_CRS)
inside = grid[grid.geometry.within(district_proj)].reset_index(drop=True)
N_SITES = len(inside)
assert N_SITES > 0, 'No candidate sites fall inside the district polygon.'

inside_wgs = inside.to_crs(GEOGRAPHIC_CRS)
df = pd.DataFrame({'site_id': [f'TALOMO_{i:04d}' for i in range(1, N_SITES + 1)],
                   'latitude': inside_wgs.geometry.y.values,
                   'longitude': inside_wgs.geometry.x.values})
site_xy = np.column_stack([inside.geometry.x.values, inside.geometry.y.values])
site_points_proj = gpd.GeoSeries([Point(x, y) for x, y in site_xy], crs=PROJECTED_CRS)
sites_wgs = gpd.GeoDataFrame(df[['site_id']].copy(),
                             geometry=[Point(x, y) for x, y in
                                       zip(df['longitude'], df['latitude'])],
                             crs=GEOGRAPHIC_CRS)

print(f'Bounding-box lattice : {len(grid):,} points at {GRID_SPACING_M} m spacing')
print(f'Inside the district  : {N_SITES:,} candidate sites '
      f'({100 * N_SITES / len(grid):.1f}% retained)')
print(f'Effective density    : {N_SITES / DISTRICT_AREA_KM2:.1f} sites per km²')
print(f'\nn = {N_SITES:,}. Replace every occurrence of 3,119 in the manuscript with '
      f'this figure; the two runs are not comparable.')

### Cell 6 — Load the OSM feature layers

Reads building footprints, the driving road network, existing health facilities, and
coastline geometry from the extract, each clipped with its own buffer around the
district. The buffers matter: a site near the district edge must still see the
footprints, roads and facilities just outside the boundary, or its features are wrong
purely because of where the administrative line was drawn.

The cell asserts that buildings, roads and facilities all loaded. Building density
carries the largest weight in the composite score, so a synthetic or partial
substitute would make every downstream number indefensible.

`RAW_OSM_RECORDS` is the count of OSM elements read. It is not the candidate site
count and the two must never be presented as comparable — that conflation is what the
export metadata's `dataset_clarification` block exists to prevent.

In [ ]:
# =============================================================================
# CELL 6 — OSM feature layers
# Reads buildings, driving roads, health facilities and coastline within buffered
# windows around the district. Asserts the three essential layers are non-empty
# and records raw element counts for the export provenance block.
# =============================================================================
def window_wgs84(buffer_m):
    return gpd.GeoSeries([district_proj.buffer(buffer_m)], crs=PROJECTED_CRS) \
             .to_crs(GEOGRAPHIC_CRS).iloc[0]


buildings_proj = roads_proj = facilities_proj = coast_proj = None
OSM_RECORD_COUNTS = {}

if OSM_SOURCE_KIND == 'pbf':
    from pyrosm import OSM
    win = window_wgs84(max(RADIUS_M, ROAD_WINDOW_M,
                           FACILITY_SEARCH_BUFFER_M, COAST_WINDOW_M)).bounds
    osm_local = OSM(OSM_SOURCE_PATH, bounding_box=list(win))
    print('Reading feature layers from the PBF within the district window...')

    b = osm_local.get_buildings()
    if b is not None and len(b):
        b = b[b.geometry.notna()].set_crs(GEOGRAPHIC_CRS, allow_override=True)
        buildings_proj = b.to_crs(PROJECTED_CRS)
        buildings_proj = buildings_proj[
            buildings_proj.geometry.intersects(district_proj.buffer(RADIUS_M))]

    r = osm_local.get_network(network_type='driving')
    if r is not None and len(r):
        r = r[r.geometry.notna()].set_crs(GEOGRAPHIC_CRS, allow_override=True)
        roads_proj = r.to_crs(PROJECTED_CRS)
        roads_proj = roads_proj[
            roads_proj.geometry.intersects(district_proj.buffer(ROAD_WINDOW_M))]

    p = osm_local.get_pois(custom_filter={'amenity': HEALTHCARE_AMENITIES})
    if p is not None and len(p):
        p = p[p.geometry.notna()].set_crs(GEOGRAPHIC_CRS, allow_override=True)
        facilities_proj = p.to_crs(PROJECTED_CRS)
        facilities_proj = facilities_proj[
            facilities_proj.geometry.intersects(district_proj.buffer(FACILITY_SEARCH_BUFFER_M))]

    try:
        n = osm_local.get_natural()
        if n is not None and len(n) and 'natural' in n.columns:
            c = n[n['natural'] == 'coastline']
            if len(c):
                c = c.set_crs(GEOGRAPHIC_CRS, allow_override=True)
                coast_proj = c.to_crs(PROJECTED_CRS)
                coast_proj = coast_proj[
                    coast_proj.geometry.intersects(district_proj.buffer(COAST_WINDOW_M))]
    except Exception as e:
        print(f'   coastline read failed: {e}')

for label, g in [('buildings', buildings_proj), ('roads', roads_proj),
                 ('facilities', facilities_proj), ('coastline', coast_proj)]:
    OSM_RECORD_COUNTS[label] = int(0 if g is None else len(g))
    print(f'   {label:11s}: {OSM_RECORD_COUNTS[label]:,} features')

assert buildings_proj is not None and len(buildings_proj), (
    'Buildings failed to load. Building density carries w₁ = 0.40; a substitute '
    'would make every downstream score indefensible.')
assert roads_proj is not None and len(roads_proj), 'Roads failed to load.'
assert facilities_proj is not None and len(facilities_proj), 'Facilities failed to load.'

RAW_OSM_RECORDS = int(sum(OSM_RECORD_COUNTS.values()))
print(f'\nRaw OSM elements read: {RAW_OSM_RECORDS:,}')
print(f'Candidate sites      : {N_SITES:,}')
print('These two figures count different things and are never comparable.')

### Cell 7 — Site attributes: features, hazards, shoreline, barangay

Computes everything that describes a candidate site, in one place:

- **Building density** — count of building-footprint centroids within `RADIUS_M`
  (300 m), by KD-tree radius query. This is the dasymetric proxy for population demand.
- **Road distance** — Euclidean distance in metres to the nearest mapped **road node**.
  Road geometry vertices are extracted and queried by KD-tree. The manuscript defines
  the metric against the nearest road *node*, so the code measures to a node, not to
  the nearest point on a segment.
- **Facility distance** — Euclidean distance to the nearest existing health facility.
  A larger value is a stronger coverage-gap signal, and is treated as positive.
- **Shoreline distance** — distance to the nearest coastline geometry, computed for
  every site and retained as a real attribute. Sites inside the 50 m setback are kept
  in the dataset with their true distance and flagged, not deleted: a deleted row
  demonstrates nothing, and the panel asked for the setback to be observable in the
  prototype.
- **Hazard classes** — ordinal class from each of the three Project NOAH layers, by
  point-in-polygon join. Source values are stored as floats and are coerced to integers;
  a site outside every polygon of a layer takes class 0; a site intersecting several
  takes the most severe.
- **Barangay** — by point-in-polygon test against the 14 resolved polygons, never by
  latitude/longitude range. Unmatched edge sites are snapped to the nearest polygon
  within one grid spacing before being written as null.

All metric computation is in EPSG:32651, so every distance printed is in true metres.

In [ ]:
# =============================================================================
# CELL 7 — Site attributes
# Computes building density, road-node distance, facility distance, shoreline
# distance, the three ordinal hazard classes, and barangay membership for every
# candidate site. All distances are metres (EPSG:32651).
# =============================================================================
# ── Building density (dasymetric population proxy) ───────────────────────────
b_cent = buildings_proj.geometry.centroid
b_xy = np.column_stack([b_cent.x.values, b_cent.y.values])
df['building_density'] = cKDTree(b_xy).query_ball_point(
    site_xy, r=RADIUS_M, return_length=True).astype(float)
print(f'Building density  ({len(b_xy):,} footprints, r={RADIUS_M} m) — '
      f'min {df.building_density.min():.0f}, max {df.building_density.max():.0f}, '
      f'mean {df.building_density.mean():.1f}')

# ── Road distance, measured to the nearest road NODE (manuscript definition) ─
try:
    from shapely import get_coordinates
    road_nodes = get_coordinates(roads_proj.geometry.values)
except Exception:
    road_nodes = np.vstack([np.asarray(g.coords) for g in roads_proj.geometry
                            if g.geom_type == 'LineString'] +
                           [np.asarray(p.coords) for g in roads_proj.geometry
                            if g.geom_type == 'MultiLineString' for p in g.geoms])
road_nodes = np.unique(road_nodes, axis=0)
d_road, _ = cKDTree(road_nodes).query(site_xy, k=1)
df['road_distance'] = d_road
print(f'Road distance     ({len(road_nodes):,} road nodes) — '
      f'min {df.road_distance.min():.1f} m, max {df.road_distance.max():.1f} m, '
      f'mean {df.road_distance.mean():.1f} m')

# ── Facility distance (coverage gap) ─────────────────────────────────────────
f_geom = facilities_proj.geometry
f_pts = f_geom if (f_geom.geom_type == 'Point').all() else f_geom.centroid
facility_xy = np.column_stack([f_pts.x.values, f_pts.y.values])
d_fac, _ = cKDTree(facility_xy).query(site_xy, k=1)
df['facility_distance'] = d_fac
print(f'Facility distance ({len(facility_xy)} facilities) — '
      f'min {df.facility_distance.min():.1f} m, max {df.facility_distance.max():.1f} m, '
      f'mean {df.facility_distance.mean():.1f} m')

# ── Shoreline distance (hard gate input; retained, never used to delete rows) ─
coast_union = None
if coast_proj is not None and len(coast_proj):
    coast_union = (coast_proj.geometry.union_all() if hasattr(coast_proj.geometry, 'union_all')
                   else coast_proj.geometry.unary_union)
else:
    try:
        import osmnx as ox
        w = window_wgs84(COAST_WINDOW_M).bounds
        c = ox.features_from_bbox(bbox=(w[0], w[1], w[2], w[3]),
                                  tags={'natural': 'coastline'})
        if len(c):
            coast_union = c.to_crs(PROJECTED_CRS).geometry.union_all()
            print(f'   Coastline recovered via OSMnx: {len(c)} features.')
    except Exception as e:
        print(f'   OSMnx coastline fallback failed: {e}')

if coast_union is not None:
    df['shoreline_distance'] = site_points_proj.distance(coast_union).values
    print(f'Shoreline distance — min {df.shoreline_distance.min():.1f} m, '
          f'median {df.shoreline_distance.median():.1f} m')
else:
    df['shoreline_distance'] = np.nan
    print('WARNING: no coastline geometry. The shoreline gate cannot be applied. '
          'Resolve before the final run.')

# ── Hazard classes ───────────────────────────────────────────────────────────
def load_noah_layer(cfg):
    g = gpd.read_file(cfg['shp'])
    g = g.set_crs(GEOGRAPHIC_CRS) if g.crs is None else g.to_crs(GEOGRAPHIC_CRS)
    fld = cfg['class_field']
    g['hazard_class'] = (pd.to_numeric(g[fld], errors='coerce').round()
                         .astype('Int64').clip(0, 3) if fld in g.columns
                         else pd.Series([0] * len(g), dtype='Int64'))
    return g[['hazard_class', 'geometry']].dropna(subset=['geometry'])


print()
HAZARD_DISTRIBUTIONS = {}
for name, cfg in NOAH_LAYERS.items():
    assert os.path.exists(cfg['shp']), f'Hazard layer missing: {cfg["shp"]}'
    layer = load_noah_layer(cfg)
    j = gpd.sjoin(sites_wgs, layer, how='left', predicate='within')
    df[name] = (j.groupby(j.index)['hazard_class'].max()
                 .reindex(range(N_SITES)).fillna(0).astype(int).values)
    d = {int(k): int(v) for k, v in df[name].value_counts().sort_index().items()}
    HAZARD_DISTRIBUTIONS[name] = d
    exposed = 100 * sum(v for k, v in d.items() if k > 0) / N_SITES
    print(f'{name:28s}: {d}  ({exposed:.1f}% exposed)')

# ── Barangay membership by point-in-polygon ──────────────────────────────────
joined = gpd.sjoin(sites_wgs, brgy_gdf[['barangay', 'geometry']],
                   how='left', predicate='within')
joined = joined[~joined.index.duplicated(keep='first')]
df['barangay'] = joined['barangay'].values

missing = df['barangay'].isna()
if missing.any():
    brgy_proj = brgy_gdf.to_crs(PROJECTED_CRS)
    for i in np.where(missing.values)[0]:
        d = brgy_proj.geometry.distance(Point(site_xy[i][0], site_xy[i][1]))
        if d.min() <= GRID_SPACING_M:
            df.loc[df.index[i], 'barangay'] = brgy_proj.loc[d.idxmin(), 'barangay']

counts = df['barangay'].value_counts()
BARANGAY_SITE_COUNTS = {b: int(counts.get(b, 0)) for b in TALOMO_BARANGAYS}
print(f'\nBarangays represented: {counts.shape[0]} of {EXPECTED_BARANGAY_COUNT}; '
      f'unmatched sites: {int(df["barangay"].isna().sum())}')
for b in TALOMO_BARANGAYS:
    print(f'   {b:20s} {BARANGAY_SITE_COUNTS[b]:5d} sites')
assert counts.shape[0] >= EXPECTED_BARANGAY_COUNT - 1, (
    'Sites resolved to fewer than 13 barangays. The district geometry or the grid '
    'is wrong — stop and inspect before scoring.')

brgy_km2 = dict(zip(brgy_gdf['barangay'], brgy_gdf['area_km2']))
brgy_bld = gpd.sjoin(
    gpd.GeoDataFrame(geometry=gpd.GeoSeries(buildings_proj.geometry.centroid, crs=PROJECTED_CRS))
       .to_crs(GEOGRAPHIC_CRS),
    brgy_gdf[['barangay', 'geometry']], how='inner', predicate='within')
BARANGAY_FOOTPRINT_DENSITY = (brgy_bld['barangay'].value_counts() / pd.Series(brgy_km2)).to_dict()
BARANGAY_FOOTPRINT_DENSITY = {b: float(BARANGAY_FOOTPRINT_DENSITY.get(b, 0.0)) for b in TALOMO_BARANGAYS}
_density_vals = np.array(list(BARANGAY_FOOTPRINT_DENSITY.values()))
_median_density = float(np.median(_density_vals[_density_vals > 0])) if (_density_vals > 0).any() else 0.0
UNDERMAPPED_THRESHOLD = 0.30 * _median_density  # below 30% of the district median
UNDERMAPPED_BARANGAYS = {b for b, d in BARANGAY_FOOTPRINT_DENSITY.items()
                         if d < UNDERMAPPED_THRESHOLD}
df['barangay_undermapped'] = df['barangay'].isin(UNDERMAPPED_BARANGAYS)

print(f'\nBuilding footprint density per barangay (footprints / km², independent of grid):')
for b in TALOMO_BARANGAYS:
    flag = '  <-- thinly mapped' if b in UNDERMAPPED_BARANGAYS else ''
    print(f'   {b:20s} {BARANGAY_FOOTPRINT_DENSITY[b]:8.1f} /km²{flag}')
print(f'   District median (mapped barangays): {_median_density:.1f} /km²   '
      f'threshold: {UNDERMAPPED_THRESHOLD:.1f} /km²')
if UNDERMAPPED_BARANGAYS:
    print(f'   {len(UNDERMAPPED_BARANGAYS)} barangay(s) flagged: {sorted(UNDERMAPPED_BARANGAYS)}')
    print('   Sites in these barangays will be marked in the export and in their '
          'interpretation text: low building-density scores there may reflect '
          'mapping coverage rather than low demand.')

# ── Quality assurance pass (manuscript: GIS-Based Feature Engineering) ───────
qa = {
    'negative distances': int((df[['road_distance', 'facility_distance']] < 0).any(axis=1).sum()),
    'null hazard classes': int(df[HAZARD_FEATURES].isna().any(axis=1).sum()),
    'sites with no barangay': int(df['barangay'].isna().sum()),
    'zero building density': int((df['building_density'] == 0).sum()),
}
print('\nQuality assurance pass (screened before normalisation):')
for k, v in qa.items():
    print(f'   {k:24s}: {v}')
df.describe().T.to_csv(os.path.join(OUTPUT_DIR, 'raw_feature_summary.csv'))
print('   raw_feature_summary.csv written.')

## STAGE 3 — SCORING

### Cell 8 — Baseline MCDA composite score

Applies, in order: **F2** to convert road distance into an accessibility value,
**F1** to normalise all six variables to [0, 1], **F3** and **F4** to aggregate them
into the hazard-integrated and service-only composites, and **F5** to rescale both
into [0, 1].

Direction is handled by the sign in **F3**, not during normalisation. All six variables
are normalised so that higher normalised value means higher raw value; service factors
are then added and hazard factors subtracted. This keeps the reported penalty directly
proportional to the hazard class, which is what the interpretation text tells the user.

The rescale bounds come only from sites outside the shoreline setback; setback sites
are clipped into [0, 1] afterwards. Retained sites therefore score identically to what
a delete-then-score pipeline would have produced, while the excluded sites stay in the
dataset and remain inspectable.

`baseline_score` is the supervised training target. `baseline_service_only` is the
ablation counterfactual: it is never trained on, never exported, and never shown.

In [ ]:
# =============================================================================
# CELL 8 — Baseline MCDA composite score
# Applies F2 (road accessibility), F1 (Min–Max normalisation), F3 (hazard-integrated
# composite), F4 (service-only counterfactual) and F5 (rescaling). Produces
# baseline_score, the supervised training target. Saves the score distribution figure.
# =============================================================================
verify_weights()   # re-asserted immediately before use

# F2 — road accessibility from distance
df['road_accessibility'] = f2_road_accessibility(df['road_distance'].values)

# F1 — Min–Max normalisation of all six variables
norm = pd.DataFrame({f: f1_minmax(df[f].values) for f in ALL_FEATURES}, index=df.index)
print('Normalised feature ranges (F1):')
print(norm.describe().T[['min', 'mean', 'max']].round(4).to_string())

# F22 (shoreline part) — the mask that defines the rescale bounds
shore_excl, _, _ = f22_eligibility(df['shoreline_distance'].values,
                                   df[HAZARD_FEATURES].values)
df['shoreline_excluded'] = shore_excl
retained_mask = ~shore_excl
N_SHORELINE_EXCLUDED = int(shore_excl.sum())

# F3 / F4 — composites, then F5 — rescale
raw_hz = f3_composite_hazard_integrated(norm)
raw_sv = f4_composite_service_only(norm)
df['raw_hazard_integrated'] = raw_hz
df['raw_service_only'] = raw_sv
df['baseline_score'], LO_HZ, HI_HZ = f5_rescale(raw_hz, retained_mask)
df['baseline_service_only'], LO_SV, HI_SV = f5_rescale(raw_sv, retained_mask)

print(f'\nF3 raw hazard-integrated : [{raw_hz.min():.4f}, {raw_hz.max():.4f}]  '
      f'(theoretical [-0.20, 0.80])')
print(f'F4 raw service-only      : [{raw_sv.min():.4f}, {raw_sv.max():.4f}]  '
      f'(theoretical [0, 0.80])')
print(f'F5 rescale bounds        : from {int(retained_mask.sum()):,} retained sites '
      f'(S_min={LO_HZ:.4f}, S_max={HI_HZ:.4f}); '
      f'{N_SHORELINE_EXCLUDED} setback site(s) clipped into [0, 1]')
print(f'Training target S′       : mean {df.baseline_score.mean():.4f}, '
      f'std {df.baseline_score.std():.4f}')

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df['baseline_score'], bins=40, color='steelblue', edgecolor='white', alpha=.85)
for t, lbl in [(TAU1, 'τ₁'), (TAU2, 'τ₂'), (TAU3, 'τ₃')]:
    ax.axvline(t, color='grey', ls=':', lw=1)
    ax.text(t, ax.get_ylim()[1] * 0.95, lbl, ha='center', fontsize=9)
ax.axvline(df['baseline_score'].mean(), color='crimson', ls='--',
           label=f'Mean = {df.baseline_score.mean():.3f}')
ax.set_title(f'Hazard-integrated baseline MCDA score\n{STUDY_AREA}, '
             f'{N_SITES:,} candidate sites')
ax.set_xlabel("Suitability score S′ (0–1)"); ax.set_ylabel('Sites'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_baseline_score_distribution.png'), dpi=150)
plt.show()

## STAGE 4 — MODEL

### Cell 9 — Partition and hyperparameter search

Splits the retained sites 80/20 and runs a grid search over the four hyperparameters
the manuscript names: number of trees `B`, maximum depth, minimum samples per split,
and number of features considered per split `m`. Selection minimises validation MSE
over five stratified folds, with `random_state = 42` fixed.

Folds are stratified by suitability band so that the sparse Highly Suitable band is
represented in every fold; bands with fewer than five members are collapsed for
stratification only, which is a mechanical requirement of `StratifiedKFold` and does
not affect the model.

Only sites outside the shoreline setback enter the partition. Their targets are
unclipped and therefore undistorted; setback sites are still scored at inference and
remain fully represented in the export.

If the search selects `max_features = 1.0`, the cell says so. The manuscript asserts
`m < k`, and that sentence would then need qualifying.

In [ ]:
# =============================================================================
# CELL 9 — Train/test partition and hyperparameter grid search
# Splits retained sites 80/20, builds five stratified CV folds, and searches the
# four hyperparameters named in the manuscript, minimising validation MSE.
# Writes model_selection.csv.
# =============================================================================
FEATURES = ALL_FEATURES
X_all = norm[FEATURES].values
y_all = df['baseline_score'].values

idx_retained = np.where(retained_mask)[0]
idx_train, idx_test = train_test_split(idx_retained, test_size=0.20,
                                       random_state=RANDOM_STATE)
X_train, y_train = X_all[idx_train], y_all[idx_train]
X_test,  y_test  = X_all[idx_test],  y_all[idx_test]

band = lambda v: 0 if v < TAU1 else 1 if v < TAU2 else 2 if v < TAU3 else 3
strat_train = np.array([band(v) for v in y_train])
_b, _c = np.unique(strat_train, return_counts=True)
_rare = set(_b[_c < 5])
strat_cv = np.array([9 if b in _rare else b for b in strat_train])
if _rare:
    print(f'Collapsed sparse band(s) {sorted(_rare)} for stratification only.')

print(f'Features : {FEATURES}')
print(f'Retained : {len(idx_retained):,} of {N_SITES:,}')
print(f'Split    : train {len(idx_train):,} (80%) / test {len(idx_test):,} (20%)')
print(f'Train band counts: {dict(zip(*np.unique(strat_train, return_counts=True)))}')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
CV_SPLITS = list(skf.split(X_train, strat_cv))

PARAM_GRID = {'n_estimators': [200, 300, 500],
              'max_depth': [None, 12, 20],
              'min_samples_split': [2, 5, 10],
              'max_features': ['sqrt', 0.5, 1.0]}
n_cfg = int(np.prod([len(v) for v in PARAM_GRID.values()]))
print(f'\nGrid: {n_cfg} configurations x {len(CV_SPLITS)} folds = '
      f'{n_cfg * len(CV_SPLITS)} fits')

search = GridSearchCV(
    estimator=RandomForestRegressor(criterion='squared_error', bootstrap=True,
                                    oob_score=True, n_jobs=-1,
                                    random_state=RANDOM_STATE),
    param_grid=PARAM_GRID, scoring='neg_mean_squared_error',
    cv=CV_SPLITS, refit=True, n_jobs=-1, verbose=1)
t0 = time.time()
search.fit(X_train, y_train)
print(f'Grid search complete in {time.time() - t0:.1f}s')

BEST_PARAMS = search.best_params_
model = search.best_estimator_
SELECTED_MODEL_NAME = 'Random Forest'
print('\nSelected configuration:')
for k, v in BEST_PARAMS.items():
    print(f'   {k:20s}: {v}')
print(f'   best CV MSE         : {-search.best_score_:.8f}')

# Per-fold stability of the selected configuration (manuscript: Tier 2)
r2s, maes, rmses = [], [], []
for tr, va in CV_SPLITS:
    m_ = RandomForestRegressor(**BEST_PARAMS, criterion='squared_error',
                               bootstrap=True, n_jobs=-1, random_state=RANDOM_STATE)
    m_.fit(X_train[tr], y_train[tr])
    p_ = m_.predict(X_train[va])
    r2s.append(f12_r2(y_train[va], p_))
    maes.append(f10_mae(y_train[va], p_))
    rmses.append(f11_rmse(y_train[va], p_))

CV_R2_MEAN, CV_R2_STD = float(np.mean(r2s)), float(np.std(r2s))
CV_MAE_MEAN, CV_MAE_STD = float(np.mean(maes)), float(np.std(maes))
CV_RMSE_MEAN, CV_RMSE_STD = float(np.mean(rmses)), float(np.std(rmses))
cv_table = pd.DataFrame({'Metric': ['MAE', 'RMSE', 'R²'],
                         'CV mean': [CV_MAE_MEAN, CV_RMSE_MEAN, CV_R2_MEAN],
                         'CV std': [CV_MAE_STD, CV_RMSE_STD, CV_R2_STD]})
print('\n5-fold stratified cross-validation (selected configuration):')
print(cv_table.round(6).to_string(index=False))

pd.DataFrame([{**BEST_PARAMS, 'cv_mae_mean': CV_MAE_MEAN, 'cv_mae_std': CV_MAE_STD,
               'cv_rmse_mean': CV_RMSE_MEAN, 'cv_rmse_std': CV_RMSE_STD,
               'cv_r2_mean': CV_R2_MEAN, 'cv_r2_std': CV_R2_STD}]) \
  .to_csv(os.path.join(OUTPUT_DIR, 'model_selection.csv'), index=False)

if BEST_PARAMS['max_features'] == 1.0:
    print('\nNOTE: max_features = 1.0 was selected, i.e. m = k. The manuscript '
          'sentence asserting m < k must be qualified.')

### Cell 10 — Evaluate, verify the equations, report importances

Three things happen here, in order.

**Evaluation.** MAE (**F10**), RMSE (**F11**), R² (**F12**) on the held-out 20 percent,
plus the out-of-bag R² (**F13**). These quantify how faithfully the ensemble reproduces
the composite score it was trained on. They do not establish that the framework picks
good sites — the target is a researcher-specified construct, so a high R² means the
surrogate fits the specification, not that the specification is right. That distinction
belongs in the Results text every time these numbers appear.

**Equation verification.** **F6** and **F7** are checked against the fitted model. The
ensemble mean is recomputed from the individual trees and compared to `model.predict`;
the variance-reduction at the root of the first tree is recomputed from the tree
structure. Both residuals are printed. This is the evidence that the manuscript's
equations describe what actually ran.

**Importances.** MDI (**F8**) and permutation importance (**F9**) for all six features,
reported beside their specified weights. Permutation importance is computed with
`scoring='neg_mean_squared_error'`, so the value is the mean increase in MSE, which is
what the manuscript defines. The scikit-learn default for a regressor is R², a different
quantity on a different scale; using the default would make the reported numbers
inconsistent with the printed formula.

In [ ]:
# =============================================================================
# CELL 10 — Held-out evaluation, equation verification, feature importances
# Computes F10–F13 on the test partition, verifies F6 and F7 against the fitted
# model, and reports F8 and F9 beside the specified weights. Saves the importance
# figure and persists the model.
# =============================================================================
pred_test = model.predict(X_test)
MAE  = f10_mae(y_test, pred_test)
RMSE = f11_rmse(y_test, pred_test)
R2   = f12_r2(y_test, pred_test)
OOB_R2 = float(model.oob_score_)

print('── Held-out performance (20% test partition)')
print(f'   F10  MAE    : {MAE:.6f}')
print(f'   F11  RMSE   : {RMSE:.6f}   (RMSE/MAE = {RMSE / MAE:.2f})')
print(f'   F12  R²     : {R2:.6f}')
print(f'   F13  OOB R² : {OOB_R2:.6f}')
print(f'        CV R²  : {CV_R2_MEAN:.6f} ± {CV_R2_STD:.6f}')
print('   Read these as fidelity to the composite-score specification, not as '
      'evidence that the specification selects good sites.')

# ── Verify the two equations scikit-learn executes internally ────────────────
print('\n── Manuscript equation verification')
resid_f6 = f6_verify_ensemble_mean(model, X_test[:200])
print(f'   F6  ŷ = (1/B)ΣTⱼ(x)  — max |manual mean − model.predict| = {resid_f6:.2e}  PASS')
split0 = f7_verify_split_criterion(model, 0)
print(f'   F7  variance-minimising split — tree 0 root splits on '
      f'{FEATURES[split0["split_feature_index"]]} at {split0["split_threshold"]:.4f}')
print(f'       parent variance {split0["parent_variance"]:.6f} → weighted child '
      f'{split0["weighted_child_variance"]:.6f} '
      f'(reduction {split0["variance_reduction"]:.6f})  PASS')

# ── Importances ──────────────────────────────────────────────────────────────
FEATURE_IMPORTANCES = f8_mdi(model, FEATURES)
PERMUTATION_IMPORTANCES, PERMUTATION_IMPORTANCE_STD = f9_permutation_importance(
    model, X_test, y_test, FEATURES, T=10)

imp_table = pd.DataFrame({
    'Feature': FEATURES,
    'Weight': [WEIGHTS[f] for f in FEATURES],
    'MDI (F8)': [FEATURE_IMPORTANCES[f] for f in FEATURES],
    'Permutation ΔMSE (F9)': [PERMUTATION_IMPORTANCES[f] for f in FEATURES],
    'Permutation SD': [PERMUTATION_IMPORTANCE_STD[f] for f in FEATURES],
}).sort_values('MDI (F8)', ascending=False)
print('\n── Feature importance, reported beside the specified weights')
print(imp_table.round(5).to_string(index=False))
print('   Permutation importance is the measure of record where the two diverge: '
      'MDI is biased toward variables with more candidate split points, and the '
      'three hazard variables are ordinal with at most three.')
imp_table.to_csv(os.path.join(OUTPUT_DIR, 'feature_importance.csv'), index=False)

pd.DataFrame([{'Model': 'Random Forest (hazard-integrated)', **BEST_PARAMS,
               'MAE': MAE, 'RMSE': RMSE, 'R2': R2, 'OOB_R2': OOB_R2,
               'CV_R2_mean': CV_R2_MEAN, 'CV_R2_std': CV_R2_STD}]) \
  .to_csv(os.path.join(OUTPUT_DIR, 'model_evaluation_metrics.csv'), index=False)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].barh(imp_table['Feature'], imp_table['MDI (F8)'], color='steelblue')
axes[0].set_title('MDI importance (F8)'); axes[0].invert_yaxis()
axes[1].barh(imp_table['Feature'], imp_table['Permutation ΔMSE (F9)'],
             xerr=imp_table['Permutation SD'], color='seagreen')
axes[1].set_title('Permutation importance (F9, held-out)'); axes[1].invert_yaxis()
axes[2].scatter(y_test, pred_test, s=6, alpha=.35, color='slategrey')
axes[2].plot([0, 1], [0, 1], 'k--', lw=1)
axes[2].set_xlabel("Baseline target S′"); axes[2].set_ylabel('Predicted ŷ')
axes[2].set_title(f'Predicted vs target (R² = {R2:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_model_evaluation.png'), dpi=150)
plt.show()

import joblib
joblib.dump(model, os.path.join(MODEL_DIR, 'polaris_rf.joblib'))
with open(os.path.join(MODEL_DIR, 'best_params.json'), 'w') as f:
    json.dump({k: (None if v is None else v) for k, v in BEST_PARAMS.items()}, f, indent=2)
print(f'\nModel and best_params.json persisted to {MODEL_DIR}')

### Cell 11 — Objective C: Random Forest against the linear weighted overlay

Specific Objective C asks how the Random Forest compares with *conventional linear
weighted-overlay approaches*. This cell answers it directly and honestly by fitting an
ordinary least-squares model on the same six normalised features and the same partition,
which is the exact functional form of a weighted linear combination.

**Expect the linear model to fit essentially perfectly.** The training target is, by
construction, a weighted linear sum of those same six features, so an OLS fit recovers
the weight vector and the residual falls to floating-point noise. The cell prints the
recovered coefficients beside the specified weights so this is visible rather than
implied.

This is not a failure of the study. It is the empirical demonstration of the surrogate
relationship the manuscript already states in the Mathematical Formalization section
and again under Limitations: no nonlinear structure can be discovered in a target that
contains none. Report it as a finding. The defensible claim is that the ensemble
reproduces the specification to within a stated tolerance while supplying the feature
importances and the generalisation to off-grid locations that the linear form does not,
and that the machinery is in place should a validated empirical target become available.

Do not report the Random Forest as outperforming the linear baseline on this target.
It does not, and the comparison table printed here will say so.

In [ ]:
# =============================================================================
# CELL 11 — Objective C comparison: Random Forest vs linear weighted overlay
# Fits OLS on the same features and partition — the functional form of a weighted
# linear combination — and tabulates both models on MAE, RMSE and R². Prints the
# recovered coefficients beside the specified weights. Writes
# objective_c_comparison.csv.
# =============================================================================
lin = LinearRegression().fit(X_train, y_train)
pred_lin = lin.predict(X_test)

comparison = pd.DataFrame([
    {'Model': 'Linear weighted overlay (OLS)', 'MAE': f10_mae(y_test, pred_lin),
     'RMSE': f11_rmse(y_test, pred_lin), 'R²': f12_r2(y_test, pred_lin)},
    {'Model': 'Random Forest (hazard-integrated)', 'MAE': MAE, 'RMSE': RMSE, 'R²': R2},
])
print('── Objective C — held-out comparison on the same partition')
print(comparison.round(6).to_string(index=False))

coef_table = pd.DataFrame({
    'Feature': FEATURES,
    'Specified weight (signed)': [WEIGHTS[f] * (1 if f in SERVICE_FEATURES else -1)
                                  for f in FEATURES],
    'OLS coefficient': lin.coef_,
})
# The specified weights operate on the raw composite; the target is rescaled by F5,
# so the OLS coefficients recover the weights scaled by 1/(S_max - S_min).
coef_table['Weight ÷ rescale span'] = (
    coef_table['Specified weight (signed)'] / (HI_HZ - LO_HZ))
print('\n── Recovered linear structure')
print(coef_table.round(5).to_string(index=False))
print(f'   Rescale span (S_max − S_min) = {HI_HZ - LO_HZ:.6f}; intercept = '
      f'{lin.intercept_:.6f}')

LINEAR_R2 = float(f12_r2(y_test, pred_lin))
print('\n── Interpretation, to be carried verbatim into the Results text')
if LINEAR_R2 > 0.999:
    print('   The linear model reproduces the target almost exactly. This is the '
          'expected and correct outcome: the target is a weighted linear sum of the '
          'same six features, so it contains no nonlinear structure for the ensemble '
          'to find. Report the Random Forest as a faithful surrogate that additionally '
          'supplies feature importances and off-grid generalisation, NOT as '
          'outperforming the linear baseline.')
else:
    print('   The linear model does not reproduce the target exactly. Investigate '
          'before reporting — under the present specification it should.')

comparison.to_csv(os.path.join(OUTPUT_DIR, 'objective_c_comparison.csv'), index=False)
coef_table.to_csv(os.path.join(OUTPUT_DIR, 'objective_c_coefficients.csv'), index=False)

## STAGE 5 — OUTPUTS

### Cell 12 — Classification, hard gates, recommendation levels

Predicts every site with the fitted model, then applies **F14** (four classes at
τ₁ = 0.25, τ₂ = 0.40, τ₃ = 0.60), **F15** (intra-class percentile → five-level
recommendation scale), and **F22** (the two hard gates).

The gates are non-compensatory and sit above the score: a site inside the 50 m shoreline
setback, or carrying class 3 on any hazard layer, is barred from recommendation no matter
how high it scores. Gated sites keep their score, their class and their real attribute
values, and carry `recommendation_eligible = false` with `recommendation_level = 0`.

The eligibility filter runs *before* the sort in Cell 14, so a gated site can never enter
a shortlist. The cell prints the highest-scoring site in the district and the highest
scoring *eligible* site; if the first is gated and the second is different, that is the
demonstration that the gate binds.

The cell also runs the Tier 2 reconstruction check: the mean absolute deviation between
the prediction and the training target, and the proportion of sites deviating by more
than 0.05.

In [ ]:
# =============================================================================
# CELL 12 — Classification, hard eligibility gates, recommendation levels
# Applies F14 (four suitability classes), F15 (intra-class percentile levels) and
# F22 (shoreline and High-hazard gates). Runs the Tier 2 reconstruction check.
# Writes polaris_site_scores.csv and the class-distribution figure.
# =============================================================================
df['score'] = np.clip(model.predict(X_all), 0.0, 1.0)
df['suitability_class'] = df['score'].apply(f14_classify)

# F15 — intra-class percentile → recommendation level
df['recommendation_level'] = 0
for cls, (lo_l, hi_l) in CLASS_LEVEL_RANGE.items():
    m_ = (df['suitability_class'] == cls).values
    if not m_.any():
        continue
    P = f15_intraclass_percentile(df.loc[m_, 'score'].values)
    df.loc[m_, 'recommendation_level'] = np.where(P >= 0.50, hi_l, lo_l)
df['recommendation_level'] = df['recommendation_level'].astype(int)

# F22 — hard gates
shore_excl, high_excl, eligible = f22_eligibility(df['shoreline_distance'].values,
                                                  df[HAZARD_FEATURES].values)
df['shoreline_excluded'] = shore_excl
df['high_hazard_excluded'] = high_excl
df['recommendation_eligible'] = eligible
df.loc[~df['recommendation_eligible'], 'recommendation_level'] = 0
df['rank'] = df['score'].rank(ascending=False, method='min').astype(int)

CLASS_COUNTS = {c: int((df['suitability_class'] == c).sum()) for c in SUITABILITY_CLASSES}
N_SHORELINE_EXCLUDED = int(shore_excl.sum())
N_HIGH_HAZARD_EXCLUDED = int(high_excl.sum())
N_ELIGIBLE = int(eligible.sum())
N_INELIGIBLE = N_SITES - N_ELIGIBLE

print('Suitability class distribution (F14):')
for c in SUITABILITY_CLASSES:
    print(f'   {c:22s} {CLASS_COUNTS[c]:6d}  ({100 * CLASS_COUNTS[c] / N_SITES:5.1f}%)')

print(f'\nHard eligibility gates (F22):')
print(f'   Shoreline setback (<{SHORELINE_BUFFER_M} m): {N_SHORELINE_EXCLUDED:,}')
print(f'   High hazard on any layer            : {N_HIGH_HAZARD_EXCLUDED:,}')
print(f'   Ineligible under either gate        : {N_INELIGIBLE:,} '
      f'({100 * N_INELIGIBLE / N_SITES:.1f}%)')
print(f'   Eligible for recommendation         : {N_ELIGIBLE:,} '
      f'({100 * N_ELIGIBLE / N_SITES:.1f}%)')

top_any = df.loc[df['score'].idxmax()]
top_elig = df[df['recommendation_eligible']].nlargest(1, 'score').iloc[0]
print(f'\nGate demonstration:')
print(f'   Highest-scoring site overall : {top_any.site_id} ({top_any.score:.3f}) — '
      f'{"ELIGIBLE" if top_any.recommendation_eligible else "GATED, correctly absent from shortlists"}')
print(f'   Highest-scoring eligible site: {top_elig.site_id} ({top_elig.score:.3f})')

dev = np.abs(df['score'].values - df['baseline_score'].values)
DRIFT_OVER_005 = float((dev > 0.05).mean())
print(f'\nTier 2 reconstruction check — mean |ŷ − S′| = {dev.mean():.4f}; '
      f'{100 * DRIFT_OVER_005:.2f}% of sites deviate by more than 0.05')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar([c.replace(' ', '\n') for c in SUITABILITY_CLASSES],
            [CLASS_COUNTS[c] for c in SUITABILITY_CLASSES],
            color=['#2e7d32', '#1565c0', '#f9a825', '#c62828'])
axes[0].set_title('Suitability class distribution'); axes[0].set_ylabel('Sites')
haz_df = pd.DataFrame({h: [HAZARD_DISTRIBUTIONS[h].get(k, 0) for k in range(4)]
                       for h in HAZARD_FEATURES}, index=list(HAZARD_LABELS.values())).T
haz_df.plot(kind='barh', stacked=True, ax=axes[1],
            color=['#bdbdbd', '#ffd54f', '#fb8c00', '#c62828'])
axes[1].set_title('Hazard class distribution by layer'); axes[1].set_xlabel('Sites')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_class_and_hazard_distribution.png'), dpi=150)
plt.show()

df.to_csv(os.path.join(OUTPUT_DIR, 'polaris_site_scores.csv'), index=False)

### Cell 13 — Plain-language interpretation text

Generates, for each site, the paragraph the application shows in three places: the
recommendation cards, the evaluation panel, and the site detail dialog.

The text is built from **that site's own** values — its normalised factors, its raw
distances in metres, its hazard classes, and its gate status. It is not built from
feature importances: those are global, identical for every site, and therefore cannot
explain why one particular site scored what it did.

No jargon reaches the string. The vocabulary is deliberately limited to what a health
officer reads without a glossary — no "composite", "normalized", "proxy", "penalty" or
"inverse distance".

The road-accessibility sentence is generated by a dedicated routine, because that number
reads backwards to anyone who has not been told: it is an inverse distance, so a value
near zero means *far from the road*. Every mention of it in the system states the
direction in words and quotes the actual distance in metres.

`SCORE_GUIDE` is the authoritative reading instruction. It is defined here, exported in
the metadata, and rendered on the welcome screen before any number is visible.

In [ ]:
# =============================================================================
# CELL 13 — Plain-language interpretation text and score guide
# Builds the per-site interpretation paragraph from that site's own factor values,
# raw distances, hazard classes and gate status. Defines SCORE_GUIDE, the
# authoritative reading instruction rendered before any score is shown.
# =============================================================================
SCORE_GUIDE = (
    'HOW TO READ SUITABILITY SCORES. Every site has one score between 0 and 1, '
    'and higher is better. The score rewards three things: how built-up the area '
    'around the site is, which stands in for how many people are there (this '
    'counts for the most); how close the site is to a road; and how far it is '
    'from the nearest health facility that already exists, since a site far from '
    'one is filling a real gap. The score then goes down where the site is '
    'exposed to flooding, landslides or storm surge. Hazard is built into every '
    'score. There is no hazard-free view of the map, because a facility that '
    'cannot stay open during a disaster fails exactly when it is needed most. '
    'Scores compare sites within Talomo District only: a score of 1.00 means the '
    'best value seen in this district, not a perfect site in any absolute sense, '
    'and the scores would need to be recalculated to compare against another '
    'district. Road access is scored so that higher means closer — a reading near '
    '0 means the site is FAR from the road network and would be hard to reach. '
    'Two rules can rule a site out no matter how well it scores: sitting within '
    '50 m of the shoreline, or carrying a High rating on flood, landslide or '
    'storm surge. Sites ruled out this way are still shown, with the reason '
    'given, so you can see what was set aside and why.')

ADVISORY_NOTE = ('POLARIS is an advisory decision-support tool; final siting '
                 'decisions remain with planning officers.')


# ── Rank bands ───────────────────────────────────────────────────────────────
# Relative wording has to match where a site really stands in the district, so each
# service factor is banded by quartile of its exported value across all sites. The
# bands go into the export and the web application's factor readings use them, so
# the interpretation and the reading beside it share one source.
# top_quartile is always the end that helps the score: most buildings, closest to a
# road, farthest from an existing facility.
UNDERSERVED_M = 1000


def rank_bands(values, higher_is_stronger=True):
    """Returns (band per site, percentile per site, cut-offs). A site's percentile is
    the share of sites it strictly beats on this factor."""
    v = np.asarray(values, dtype=float)
    q25, q50, q75 = (float(q) for q in np.percentile(v, [25, 50, 75]))
    if higher_is_stronger:
        band = lambda x: ('top_quartile' if x >= q75 else 'upper_middle' if x >= q50
                          else 'lower_middle' if x >= q25 else 'bottom_quartile')
        strength = v
    else:
        band = lambda x: ('bottom_quartile' if x >= q75 else 'lower_middle' if x >= q50
                          else 'upper_middle' if x >= q25 else 'top_quartile')
        strength = -v
    pct = [round(100.0 * float(np.mean(strength < s)), 1) for s in strength]
    return [band(x) for x in v], pct, {'p25': q25, 'p50': q50, 'p75': q75}


# Banded on the values as exported, so any reader can reproduce them from the JSON.
df['band_buildings'], df['pct_buildings'], _cut_b = rank_bands(df['building_density'].astype(int))
df['band_road'], df['pct_road'], _cut_r = rank_bands(
    [round(float(v), 1) for v in df['road_distance']], higher_is_stronger=False)
df['band_facility'], df['pct_facility'], _cut_f = rank_bands(
    [round(float(v), 1) for v in df['facility_distance']])
RANK_CUTOFFS = {'building_density': _cut_b, 'road_distance_m': _cut_r, 'facility_distance_m': _cut_f}


def road_sentence(band, raw_d):
    """States the direction of the road reading in words, every time."""
    if band == 'bottom_quartile':
        return (f'This is one of the farthest sites from any road in the district '
                f'({raw_d:,.0f} m to the nearest road), so it would be hard to reach.')
    if band == 'lower_middle':
        return (f'Road access is weak — about {raw_d:,.0f} m to the nearest road, '
                f'farther than most of the district.')
    if band == 'upper_middle':
        return f'Road access is moderate, roughly {raw_d:,.0f} m to the nearest road.'
    return f'Road access is strong — about {raw_d:,.0f} m to the nearest road.'


def build_interpretation(row, norm_row):
    parts = []
    bd = row.band_buildings
    if bd == 'top_quartile':
        parts.append(f'The area around this site is more built up than most of the '
                     f'district ({int(row.building_density)} buildings within {RADIUS_M} m), '
                     f'which points to higher demand.')
    elif bd == 'upper_middle':
        parts.append(f'The surrounding area is moderately built up '
                     f'({int(row.building_density)} buildings within {RADIUS_M} m).')
    else:
        parts.append(f'Few buildings surround this site '
                     f'({int(row.building_density)} within {RADIUS_M} m), so demand '
                     f'here is likely low.')
    # PATCH — mapping-completeness caveat. Where the surrounding barangay is
    # thinly mapped in OpenStreetMap (see Cell 7), a low building-density
    # reading may reflect map coverage rather than actual low demand. Stated
    # only where it applies, and only once per site.
    if bool(row.get('barangay_undermapped', False)):
        parts.append('Note: this area has fewer mapped buildings than most of the '
                     'district, so this count may understate how built-up it really '
                     'is rather than reflect genuinely low demand.')
    parts.append(road_sentence(row.band_road, row.road_distance))
    if row.facility_distance >= UNDERSERVED_M:
        parts.append(f'The nearest existing health facility is about '
                     f'{row.facility_distance:,.0f} m away, so this area is '
                     f'currently underserved.')
    else:
        parts.append(f'An existing health facility is already about '
                     f'{row.facility_distance:,.0f} m away.')
    hz = [(h.replace('_susceptibility', '').replace('_', ' '), int(row[h]))
          for h in HAZARD_FEATURES]
    high = [n for n, v in hz if v == 3]
    mod = [n for n, v in hz if v == 2]
    if high:
        parts.append(f'Hazard rating is High for {", ".join(high)}.')
    elif mod:
        parts.append(f'Hazard rating is Moderate for {", ".join(mod)}.')
    elif all(v == 0 for _, v in hz):
        parts.append('No flood, landslide or storm-surge hazard is mapped here.')
    else:
        parts.append('Hazard exposure here is low.')
    if not row.recommendation_eligible:
        if row.shoreline_excluded:
            parts.append(f'This site cannot be recommended: it lies within '
                         f'{SHORELINE_BUFFER_M} m of the shoreline '
                         f'({row.shoreline_distance:,.0f} m).')
        else:
            parts.append('This site cannot be recommended because it carries a High '
                         'hazard rating.')
    return ' '.join(parts)


df['interpretation'] = [build_interpretation(df.iloc[i], norm.iloc[i])
                        for i in range(len(df))]
print(f'Interpretation text generated for {len(df):,} sites.\n')

_best = df[df['recommendation_eligible']].nlargest(1, 'score').iloc[0]
print(f'Example — highest-scoring eligible site ({_best.site_id}, '
      f'{_best.barangay}, score {_best.score:.3f}):')
print('   ' + _best.interpretation)
_gated = df[~df['recommendation_eligible']].nlargest(1, 'score')
if len(_gated):
    g = _gated.iloc[0]
    print(f'\nExample — highest-scoring GATED site ({g.site_id}, score {g.score:.3f}):')
    print('   ' + g.interpretation)

### Cell 14 — Export contract and acceptance checklist

Assembles `candidate_sites.json`, the only file the web application reads, and runs an
acceptance checklist against it before writing.

The schema is a contract. The nested `factors`, `factors_normalized` and `hazards`
objects are read directly by the application; flattening any of them breaks it at
runtime. The checklist verifies structure, value ranges, gate consistency, barangay
coverage, and that the exported weights still equal `MANUSCRIPT_WEIGHTS`. It **raises**
on any failure — a file that fails the checklist is not deployed.

`metadata.model_status` is deliberately never written. Its presence makes the
application render a "Provisional dataset" badge, and these are real Random Forest
predictions.

The cell also writes the four supporting GeoJSON layers — district boundary, barangay
polygons, existing facilities, coastline — from the same run as the scores, so the map
and the numbers can never come from different vintages.

After running, copy the contents of the `webapp/` folder into `polaris/public/data/`.
Do not run `npm run data:build` afterwards: the superseded migration script shares that
output path.

In [ ]:
# =============================================================================
# CELL 14 — Export contract, acceptance checklist, supporting map layers
# Builds candidate_sites.json to the schema the web application expects, runs a
# pass/fail acceptance checklist, and writes the boundary, barangay, facility and
# coastline GeoJSON layers from the same run. Raises if any check fails.
# =============================================================================
r4 = lambda v: None if v is None or (isinstance(v, float) and np.isnan(v)) else round(float(v), 4)
r1 = lambda v: None if v is None or (isinstance(v, float) and np.isnan(v)) else round(float(v), 1)


def build_site_record(row, norm_row):
    """One site record. The nested factors / factors_normalized / hazards objects
    are the contract; suitability.ts derives the shoreline gate from
    factors.shoreline_distance_m and the High-hazard gate from hazards[key]."""
    brgy = row['barangay']
    brgy = None if (brgy is None or (not isinstance(brgy, str) and pd.isna(brgy))) else str(brgy)
    return {
        'site_id': str(row['site_id']),
        'latitude': r4(row['latitude']),
        'longitude': r4(row['longitude']),
        'barangay': brgy,
        'score': r4(row['score']),
        'suitability_class': str(row['suitability_class']),
        'recommendation_level': int(row['recommendation_level']),
        'recommendation_eligible': bool(row['recommendation_eligible']),
        'interpretation': str(row['interpretation']),
        'factors': {
            'building_density': int(row['building_density']),
            'road_distance_m': r1(row['road_distance']),
            'facility_distance_m': r1(row['facility_distance']),
            'shoreline_distance_m': r1(row['shoreline_distance']),
        },
        'factors_normalized': {f: r4(norm_row[f]) for f in ALL_FEATURES},
        'hazards': {h: int(row[h]) for h in HAZARD_FEATURES},
        # Where the site stands in the district on each service factor. The interpretation
        # is written from these and the application's factor readings use them too.
        'rank_bands': {
            'building_density': {'band': str(row['band_buildings']),
                                 'percentile': float(row['pct_buildings'])},
            'road_accessibility': {'band': str(row['band_road']),
                                   'percentile': float(row['pct_road'])},
            'facility_distance': {'band': str(row['band_facility']),
                                  'percentile': float(row['pct_facility']),
                                  'underserved': bool(row['facility_distance'] >= UNDERSERVED_M)},
        },
        # Additive fields — available to the UI, not required by it.
        'recommendation_label': LEVEL_LABELS[int(row['recommendation_level'])],
        'shoreline_excluded': bool(row['shoreline_excluded']),
        'high_hazard_excluded': bool(row['high_hazard_excluded']),
        'hazard_labels': {h: HAZARD_LABELS[int(row[h])] for h in HAZARD_FEATURES},
        'rank': int(row['rank']),
        'barangay_undermapped': bool(row.get('barangay_undermapped', False)),
    }


records = [build_site_record(df.iloc[i], norm.iloc[i]) for i in range(len(df))]
by_id = {r['site_id']: r for r in records}
print(f'Built {len(records):,} site records.')

# Eligibility filter runs BEFORE the sort, so a gated site can never appear here.
eligible_sorted = df[df['recommendation_eligible']].sort_values('score', ascending=False)
top_recommendations = [{
    'recommendation_rank': rk,
    'site_id': by_id[sid]['site_id'], 'score': by_id[sid]['score'],
    'suitability_class': by_id[sid]['suitability_class'],
    'latitude': by_id[sid]['latitude'], 'longitude': by_id[sid]['longitude'],
    'barangay': by_id[sid]['barangay'], 'interpretation': by_id[sid]['interpretation'],
    'recommendation_eligible': by_id[sid]['recommendation_eligible'], # Added this line
} for rk, sid in enumerate(eligible_sorted['site_id'].head(TOP_N_RECOMMENDATIONS), 1)]
print('Shortlist: ' + ', '.join(f'#{r["recommendation_rank"]} {r["site_id"]} '
                                f'({r["score"]:.3f})' for r in top_recommendations))

metadata = {
    # ── Read by the deployed application. Do not rename. ────────────────────
    'project': 'POLARIS',
    'study_area': f'{STUDY_AREA}, Davao City',
    'model': 'Random Forest regressor (hazard-integrated, single configuration)',
    'score_guide': SCORE_GUIDE,
    'thresholds': {'tau1': TAU1, 'tau2': TAU2, 'tau3': TAU3},
    'weights': {'service': dict(SERVICE_WEIGHTS),
                'hazard_penalty': dict(HAZARD_PENALTY_WEIGHTS)},
    'shoreline_buffer_m': int(SHORELINE_BUFFER_M),
    'top_n_recommendations': int(TOP_N_RECOMMENDATIONS),
    'dataset_clarification': {
        'raw_osm_records_extracted': int(RAW_OSM_RECORDS),
        'training_candidate_sites': int(N_SITES),
        'shoreline_excluded_sites': int(N_SHORELINE_EXCLUDED),
        'note': ('Raw OSM records are the map features gathered to measure each site '
                 '— building footprints, road segments, existing health facilities '
                 'and coastline. Candidate sites are the grid points scored by the '
                 'model. The two figures count different things and are not '
                 'comparable.'),
    },
    'advisory_note': ADVISORY_NOTE,
    'generated_by': 'POLARIS_model.ipynb',
    'generated_at': datetime.now(timezone.utc).isoformat(),
    # ── Additive provenance. Not required by the UI; keeps results traceable. ─
    'barangays': list(TALOMO_BARANGAYS),
    'barangay_site_counts': BARANGAY_SITE_COUNTS,
    'barangay_footprint_density_per_km2': BARANGAY_FOOTPRINT_DENSITY,
    'undermapped_barangays': sorted(UNDERMAPPED_BARANGAYS),
    'osm_snapshot_date': OSM_SNAPSHOT_DATE,
    'osm_record_counts': OSM_RECORD_COUNTS,
    'grid_spacing_m': int(GRID_SPACING_M),
    'building_density_radius_m': int(RADIUS_M),
    'rank_bands': {'method': 'Quartiles of the exported value across all candidate sites.',
                   'cutoffs': RANK_CUTOFFS},
    'district_area_km2': round(DISTRICT_AREA_KM2, 2),
    'high_hazard_excluded_sites': int(N_HIGH_HAZARD_EXCLUDED),
    'eligible_sites': int(N_ELIGIBLE),
    'suitability_class_counts': CLASS_COUNTS,
    'hazard_class_distributions': HAZARD_DISTRIBUTIONS,
    'model_detail': {'estimator': SELECTED_MODEL_NAME,
                     'best_params': {k: v for k, v in BEST_PARAMS.items()},
                     'random_state': int(RANDOM_STATE),
                     'train_sites': int(len(idx_train)),
                     'test_sites': int(len(idx_test))},
    'validation': {'mae': MAE, 'rmse': RMSE, 'r2': R2, 'oob_r2': OOB_R2,
                   'cv_r2_mean': CV_R2_MEAN, 'cv_r2_std': CV_R2_STD,
                   'linear_overlay_r2': LINEAR_R2},
    'feature_importances': FEATURE_IMPORTANCES,
    'permutation_importances': PERMUTATION_IMPORTANCES,
}

payload = {'metadata': metadata, 'top_recommendations': top_recommendations,
           'sites': records}

# ── Acceptance checklist ─────────────────────────────────────────────────────
issues = []


def chk(cond, label):
    print(f'   [{"PASS" if cond else "FAIL"}]  {label}')
    if not cond:
        issues.append(label)


REQUIRED_SITE_KEYS = ['site_id', 'latitude', 'longitude', 'barangay', 'score',
                      'suitability_class', 'recommendation_level',
                      'recommendation_eligible', 'interpretation', 'factors',
                      'factors_normalized', 'hazards']
REQUIRED_FACTOR_KEYS = ['building_density', 'road_distance_m',
                        'facility_distance_m', 'shoreline_distance_m']
REQUIRED_META_KEYS = ['project', 'study_area', 'model', 'score_guide', 'thresholds',
                      'weights', 'shoreline_buffer_m', 'top_n_recommendations',
                      'dataset_clarification', 'advisory_note']

print('\n' + '=' * 70 + '\n  ACCEPTANCE CHECKLIST\n' + '=' * 70)
chk(all(k in payload for k in ('metadata', 'top_recommendations', 'sites')),
    'payload has metadata / top_recommendations / sites')
chk(all(all(k in r for k in REQUIRED_SITE_KEYS) for r in records),
    'every site carries all required top-level keys')
chk(all(all(k in r['factors'] for k in REQUIRED_FACTOR_KEYS) for r in records),
    'factors carries all four raw values including shoreline_distance_m')
chk(all(set(r['factors_normalized']) == set(ALL_FEATURES) for r in records),
    'factors_normalized carries all six normalised features')
chk(all(set(r['hazards']) == set(HAZARD_FEATURES) for r in records),
    'hazards carries all three layers')
chk(all(k in metadata for k in REQUIRED_META_KEYS),
    'metadata carries every key the application reads')
chk(set(metadata['weights']) == {'service', 'hazard_penalty'},
    'metadata.weights nested as service / hazard_penalty')
chk('model_status' not in metadata,
    'metadata.model_status absent (no provisional badge)')
chk(all(set(r['rank_bands']) == {'building_density', 'road_accessibility', 'facility_distance'}
        for r in records), 'every site carries its three rank bands')
chk(all(('one of the farthest' in r['interpretation'])
        == (r['rank_bands']['road_accessibility']['band'] == 'bottom_quartile') for r in records),
    '"one of the farthest" is used for the farthest quartile only')
chk(not any(r['interpretation'].startswith('Few buildings')
            and r['rank_bands']['building_density']['band'] == 'top_quartile' for r in records),
    '"few buildings" is never used in the top quartile')
chk(not any(v is None for r in records for v in
            (r['score'], r['suitability_class'], r['interpretation'])),
    'no null score, class or interpretation')
chk(all(r['suitability_class'] in SUITABILITY_CLASSES for r in records),
    'suitability_class values all valid')
chk(all(0.0 <= r['score'] <= 1.0 for r in records), 'every score within [0, 1]')
chk(all(v in (0, 1, 2, 3) for r in records for v in r['hazards'].values()),
    'hazard classes all ordinal 0–3')
_brgy = {r['barangay'] for r in records if r['barangay']}
chk(len(_brgy) >= EXPECTED_BARANGAY_COUNT - 1,
    f'sites span {len(_brgy)} barangays (expected {EXPECTED_BARANGAY_COUNT})')
chk(sum(1 for r in records if r['shoreline_excluded']) == N_SHORELINE_EXCLUDED,
    f'shoreline-excluded sites retained and counted ({N_SHORELINE_EXCLUDED})')
chk(all(r['recommendation_eligible'] for r in top_recommendations),
    'no ineligible site in top_recommendations')
chk(all(r['recommendation_level'] == 0 for r in records
        if not r['recommendation_eligible']),
    'every ineligible site carries recommendation_level 0')
chk(abs(sum(metadata['weights']['service'].values()) - 0.80) < 1e-9 and
    abs(sum(metadata['weights']['hazard_penalty'].values()) - 0.20) < 1e-9,
    'exported weights sum to 0.80 / 0.20')
chk(all(abs({**metadata['weights']['service'],
             **metadata['weights']['hazard_penalty']}[f] - MANUSCRIPT_WEIGHTS[f]) < 1e-9
        for f in ALL_FEATURES),
    'exported weights equal MANUSCRIPT_WEIGHTS')

# Halt BEFORE any file is touched. A failed check must leave the previous export
# exactly as it was on disk.
if issues:
    raise AssertionError('Export failed the acceptance checklist: ' + '; '.join(issues))
print('\n  No validation issues.')

# ── Write ────────────────────────────────────────────────────────────────────
out_path = os.path.join(WEBAPP_DIR, 'candidate_sites.json')
for p in (out_path, os.path.join(OUTPUT_DIR, 'candidate_sites.json')):
    with open(p, 'w') as f:
        json.dump(payload, f, separators=(',', ':'))
print(f'\n   candidate_sites.json — {os.path.getsize(out_path) / 1e6:.2f} MB')

gpd.GeoDataFrame({'name': [f'{STUDY_AREA}, Davao City'],
                  'source': ['Union of 14 OSM admin_level=10 barangay polygons']},
                 geometry=[district_poly], crs=GEOGRAPHIC_CRS) \
   .to_file(os.path.join(WEBAPP_DIR, 'talomo_boundary.json'), driver='GeoJSON')
brgy_gdf[['barangay', 'geometry']].rename(columns={'barangay': 'name'}) \
   .to_file(os.path.join(WEBAPP_DIR, 'talomo_barangays.json'), driver='GeoJSON')

# Detail fields behind the web app's facility dialog. They describe the facility
# only; none of them reaches a score. pyrosm gives common tags their own columns
# and packs the rest into a JSON `tags` column, so both are merged here.
def facility_tags(row):
    t = {}
    raw = row.get('tags')
    if isinstance(raw, str):
        try:
            t.update(json.loads(raw))
        except ValueError:
            pass
    elif isinstance(raw, dict):
        t.update(raw)
    for k, v in row.items():
        if k not in ('geometry', 'tags') and isinstance(v, str) and v.strip():
            t[k] = v
    return t


def facility_details(tags):
    """Readable detail fields for one facility from its raw OSM tags."""
    def first(*keys):
        for k in keys:
            v = tags.get(k)
            if isinstance(v, str) and v.strip():
                return v.strip()
        return None

    name = first('name')
    aliases = []
    for k in ('official_name', 'alt_name', 'short_name', 'old_name'):
        for v in (first(k) or '').split(';'):
            v = v.strip()
            if v and v != name and v not in aliases:
                aliases.append(v)

    street = ' '.join(p for p in (first('addr:housenumber'), first('addr:street')) if p)
    local = [p for p in (
        first('addr:unit'), first('addr:floor') and f"{first('addr:floor')} floor",
        first('addr:housename'), street or None,
        first('addr:place', 'addr:neighbourhood', 'addr:village', 'addr:quarter'),
        first('addr:barangay')) if p]
    # City and postcode alone locate nothing, so they never make an address.
    address = ', '.join(local + [p for p in (first('addr:city'), first('addr:postcode')) if p])         if local else None

    return {
        'also_known_as': '; '.join(aliases) or None,
        'speciality': first('healthcare:speciality'),
        'operator': first('operator'),
        'operator_type': first('operator:type'),
        'address': address,
        'phone': first('contact:phone', 'phone', 'contact:mobile', 'mobile'),
        'email': first('contact:email', 'email'),
        'website': first('contact:website', 'website'),
        'facebook': first('contact:facebook', 'facebook'),
        'opening_hours': first('opening_hours'),
        'beds': first('beds'),
        'emergency': first('emergency'),
        'wheelchair': first('wheelchair'),
    }


_fac_wgs = gpd.GeoSeries(f_pts, crs=PROJECTED_CRS).to_crs(GEOGRAPHIC_CRS)
_kind = (facilities_proj['amenity'].fillna('facility').values
         if 'amenity' in facilities_proj.columns else ['facility'] * len(_fac_wgs))
_fac_details = pd.DataFrame([facility_details(facility_tags(r))
                             for _, r in facilities_proj.iterrows()])
gpd.GeoDataFrame({'name': (facilities_proj['name'].fillna('Unnamed facility').values
                           if 'name' in facilities_proj.columns
                           else ['Unnamed facility'] * len(_fac_wgs)),
                  'amenity': _kind,
                  'kind': [str(k).replace('_', ' ').title() for k in _kind],
                  'osm_type': (facilities_proj['osm_type'].values
                               if 'osm_type' in facilities_proj.columns else None),
                  'osm_id': (facilities_proj['id'].values
                             if 'id' in facilities_proj.columns else None),
                  **{k: _fac_details[k].values for k in _fac_details.columns}},
                 geometry=list(_fac_wgs), crs=GEOGRAPHIC_CRS) \
   .to_file(os.path.join(WEBAPP_DIR, 'talomo_facilities.json'), driver='GeoJSON')

if coast_union is not None:
    gpd.GeoDataFrame({'natural': ['coastline']},
                     geometry=[gpd.GeoSeries([coast_union], crs=PROJECTED_CRS)
                               .to_crs(GEOGRAPHIC_CRS).iloc[0]], crs=GEOGRAPHIC_CRS) \
       .to_file(os.path.join(WEBAPP_DIR, 'talomo_coastline.json'), driver='GeoJSON')

print('\n' + '=' * 70 + f'\n  WEB APP ASSETS — {WEBAPP_DIR}\n' + '=' * 70)
for fn in sorted(os.listdir(WEBAPP_DIR)):
    print(f'    {fn:28s} {os.path.getsize(os.path.join(WEBAPP_DIR, fn)) / 1e6:6.2f} MB')
print('\n  Copy these into polaris/public/data/, replacing the existing files.')
print('  Do NOT run `npm run data:build` afterwards — the superseded migration')
print('  script shares that output path.')


## STAGE 6 — VALIDATION AND REPORTING

### Cell 15 — Internal hazard sensitivity ablation

Quantifies what hazard integration does to the scores, using **F16** (Spearman ρ),
**F17** (Classification Change Percentage), **F18** (mean deviation), **F19** (top-N
Jaccard) and **F20** (per-site spatial difference).

The comparison is against a *counterfactual*, not a second trained model: the
service-only composite is computed arithmetically, never fitted, never exported, and
never shown to a user. Magnitude measures (F18, F20) run on the raw pre-rescaling
scores, because each configuration is rescaled against its own minimum and maximum and
any level shift would otherwise be absorbed by the transformation. Rank and class
measures (F16, F17, F19) are invariant to that rescaling.

**Report the two mechanisms separately.** ρ and C describe the graduated penalty only.
The hard gates sit outside this comparison entirely and bar a far larger share of the
candidate set. Presenting ρ and C alone understates the role of hazard in the system.

In [ ]:
# =============================================================================
# CELL 15 — Internal hazard sensitivity ablation
# Applies F16–F20 to compare the hazard-integrated configuration against the
# service-only counterfactual. Magnitude measures use raw pre-rescaling scores.
# Writes internal_sensitivity_summary.json and the three-panel figure.
# =============================================================================
s_hz_raw, s_sv_raw = df['raw_hazard_integrated'].values, df['raw_service_only'].values
s_hz, s_sv = df['baseline_score'].values, df['baseline_service_only'].values

# F18 / F20 — magnitude, on RAW scores
delta_raw = f20_spatial_difference(s_sv_raw, s_hz_raw)
MEAN_DEVIATION = f18_mean_deviation(s_sv_raw, s_hz_raw)
MEAN_ABS_DEV, MAX_ABS_DEV = float(np.abs(delta_raw).mean()), float(np.abs(delta_raw).max())

# F16 — rank agreement
rank_hz = pd.Series(s_hz).rank(ascending=False, method='min').values
rank_sv = pd.Series(s_sv).rank(ascending=False, method='min').values
SPEARMAN_RHO, P_RHO, RHO_CLOSED = f16_spearman(rank_sv, rank_hz)

# F17 — classification change
sv_shared, hz_shared, _lo_shared, _hi_shared = f5_shared_rescale(
    s_sv_raw, s_hz_raw, retained_mask)
class_hz = pd.Series(hz_shared).apply(f14_classify).values
class_sv = pd.Series(sv_shared).apply(f14_classify).values
CLASSIFICATION_CHANGE_PCT, N_CHANGED = f17_classification_change_pct(class_sv, class_hz)
print(f'   (F17 computed on shared-scale bounds [{_lo_shared:.4f}, {_hi_shared:.4f}]; '
      f'independently-rescaled s_hz/s_sv from Cell 8 are NOT used for this measure.)')

print(f'F18  Δ̄  mean deviation      = {MEAN_DEVIATION:.6f}  '
      f'(|Δ| mean {MEAN_ABS_DEV:.6f}, max {MAX_ABS_DEV:.6f})   [raw scores]')
print(f'F16  ρ  Spearman            = {SPEARMAN_RHO:.6f}  (p = {P_RHO:.2e})')
print(f'        closed-form ρ (no tie correction) = {RHO_CLOSED:.6f}')
print(f'F17  C  classification change = {CLASSIFICATION_CHANGE_PCT:.2f}%  '
      f'({N_CHANGED:,} of {N_SITES:,} sites)')

JACCARD = {}
print('\nF19  Top-N Jaccard similarity')
for N in (5, 10, 20, 50):
    j, overlap = f19_jaccard_topn(s_sv, s_hz, N)
    JACCARD[N] = j
    print(f'     N = {N:<4d} overlap {overlap:<4d} J = {j:.4f}')

print(f'\nGate mechanism — OUTSIDE this comparison entirely')
print(f'     {N_INELIGIBLE:,} of {N_SITES:,} sites ({100 * N_INELIGIBLE / N_SITES:.1f}%) '
      f'are barred from recommendation by a hard gate.')
print( '     Report the gates separately from ρ and C. The two mechanisms are not '
       'commensurable, and ρ and C alone understate the role of hazard.')

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].hist(delta_raw, bins=40, color='indianred', edgecolor='white')
axes[0].set_title('F20 — per-site displacement Δᵢ (raw)')
axes[0].set_xlabel('service-only − hazard-integrated')
axes[1].scatter(s_sv, s_hz, s=4, alpha=.3, color='slategrey')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('Service-only (rescaled)'); axes[1].set_ylabel('Hazard-integrated (rescaled)')
axes[1].set_title(f'Rescaled scores (ρ = {SPEARMAN_RHO:.4f})')
sc = axes[2].scatter(df['longitude'], df['latitude'], c=delta_raw, s=5, cmap='Reds')
axes[2].set_title('F20 — spatial difference mapping'); plt.colorbar(sc, ax=axes[2])
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig_internal_sensitivity.png'), dpi=150)
plt.show()

with open(os.path.join(OUTPUT_DIR, 'internal_sensitivity_summary.json'), 'w') as f:
    json.dump({'mean_deviation_raw': MEAN_DEVIATION,
               'mean_abs_deviation_raw': MEAN_ABS_DEV,
               'max_abs_deviation_raw': MAX_ABS_DEV,
               'spearman_rho': SPEARMAN_RHO,
               'spearman_rho_closed_form': RHO_CLOSED,
               'classification_change_pct': CLASSIFICATION_CHANGE_PCT,
               'sites_changed': N_CHANGED,
               'jaccard': {str(k): v for k, v in JACCARD.items()},
               'gated_sites': N_INELIGIBLE,
               'note': 'Internal validation artefact. Never exported to the application.'},
              f, indent=2)

### Cell 16 — Population proxy against the PSA 2024 census

Compares building-footprint counts per barangay against the official PSA 2024 POPCEN
barangay counts, and reports the margin of error of the population proxy.

Three things make this join work where an earlier attempt returned zero. It uses the
**same normaliser** as the boundary matcher from Cell 4, so `Catalunan Pequeño`,
`Catalunan Pequeno` and `CATALUNAN PEQUENO` land together. It counts building centroids
by point-in-polygon against the barangay polygons, so no footprint is double-counted
across overlapping site radii. And it reports **four separate measures** — Pearson r,
Spearman ρ, mean absolute proportional deviation, and signed bias — because a single
accuracy figure is not an adequate validation of a disaggregation proxy.

If the census workbook is absent, the cell prints `PENDING` and substitutes nothing.
No estimated population figure is ever written.

The signed bias is the number to carry into Limitations: prior work using
two-dimensional building measures reports underestimation in dense areas and
overestimation in sparse ones, and this cell measures whether that pattern holds here.

In [ ]:
# =============================================================================
# CELL 16 — Population proxy margin-of-error assessment (Tier 3)
# Joins building-footprint counts per barangay to PSA 2024 POPCEN counts using the
# Cell 4 normaliser, and reports Pearson r, Spearman rho, mean absolute deviation
# and signed bias separately. Reports PENDING and substitutes nothing if the
# census source is unavailable.
# =============================================================================
PSA_FILE = '/content/drive/MyDrive/polaris_Thesis/Region XI_0.xlsx'
PSA_SHEET = None                          # None = search every sheet
PSA_MANUAL = {
    'Bago Aplaya': 18515, 'Bago Gallera': 21409, 'Baliok': 18118,
    'Bucana': 83192, 'Catalunan Grande': 42868, 'Catalunan Pequeño': 29445,
    'Dumoy': 21900, 'Langub': 4597, 'Ma-a': 61565, 'Magtuod': 5220,
    'Matina Aplaya': 33531, 'Matina Crossing': 39797, 'Matina Pangi': 22277,
    'Talomo': 65379,
}

# ── Proxy: building footprints per barangay, no double counting ──────────────
b_cent_gdf = gpd.GeoDataFrame(geometry=gpd.GeoSeries(b_cent, crs=PROJECTED_CRS)) \
                .to_crs(GEOGRAPHIC_CRS)
bj = gpd.sjoin(b_cent_gdf, brgy_gdf[['barangay', 'geometry']],
               how='inner', predicate='within')
PROXY_COUNTS = bj['barangay'].value_counts().to_dict()
print('Building footprints per barangay (point-in-polygon, no double counting):')
for b in TALOMO_BARANGAYS:
    print(f'   {b:20s} {int(PROXY_COUNTS.get(b, 0)):7d}')
print(f'   {"TOTAL":20s} {int(sum(PROXY_COUNTS.values())):7d}')


def load_psa(path, sheet=None):
    """Scan every sheet and column for barangay names, using the SAME normaliser
    as the boundary matcher, and take the first plausible numeric cell to the right."""
    if not os.path.exists(path):
        return None
    sheets = pd.read_excel(path, sheet_name=sheet if sheet is not None else None,
                           header=None)
    if isinstance(sheets, pd.DataFrame):
        sheets = {'0': sheets}
    best = {}
    for sname, raw in sheets.items():
        raw = raw.astype(object)
        for ci in range(raw.shape[1]):
            hits = {}
            for ri, val in raw.iloc[:, ci].items():
                if not isinstance(val, str):
                    continue
                for canon in TALOMO_BARANGAYS:
                    if name_matches(canon, val)[0]:
                        for cj in range(ci + 1, min(ci + 8, raw.shape[1])):
                            v = pd.to_numeric(raw.iloc[ri, cj], errors='coerce')
                            if pd.notna(v) and v > 100:
                                hits[canon] = int(v)
                                break
                        break
            if len(hits) > len(best):
                best = hits
                print(f'   sheet "{sname}" col {ci}: matched {len(hits)} barangays')
    return best or None


CENSUS = dict(PSA_MANUAL) if PSA_MANUAL else (load_psa(PSA_FILE, PSA_SHEET) or {})
PROXY_VALIDATION_STATUS = 'PENDING'
proxy_table = None
PEARSON_R = SPEARMAN_PROXY = MEAN_ABS_DEV_PCT = SIGNED_BIAS_PCT = None

if len(CENSUS) < 10:
    print(f'\nPENDING — only {len(CENSUS)} of {EXPECTED_BARANGAY_COUNT} barangays '
          f'resolved from the census source.')
    print('   Upload the PSA 2024 POPCEN barangay file to PSA_FILE, or transcribe the '
          'official counts into PSA_MANUAL. No figure is substituted.')
else:
    rows = [{'Barangay': b, 'PSA_2024_Population': CENSUS[b],
             'Building_Footprints': int(PROXY_COUNTS.get(b, 0))}
            for b in TALOMO_BARANGAYS if b in CENSUS]
    proxy_table = pd.DataFrame(rows)
    tot_pop = proxy_table['PSA_2024_Population'].sum()
    tot_prox = proxy_table['Building_Footprints'].sum()
    k = tot_pop / tot_prox if tot_prox else np.nan
    proxy_table['Persons_per_Footprint'] = (
        proxy_table['PSA_2024_Population'] / proxy_table['Building_Footprints']).round(2)
    proxy_table['Proxy_Implied_Population'] = (
        proxy_table['Building_Footprints'] * k).round(0)
    proxy_table['Signed_Deviation_Pct'] = (
        100 * (proxy_table['Proxy_Implied_Population'] - proxy_table['PSA_2024_Population'])
        / proxy_table['PSA_2024_Population']).round(2)
    proxy_table['Abs_Deviation_Pct'] = proxy_table['Signed_Deviation_Pct'].abs()

    PEARSON_R = float(pearsonr(proxy_table['Building_Footprints'],
                               proxy_table['PSA_2024_Population'])[0])
    SPEARMAN_PROXY = float(spearmanr(proxy_table['Building_Footprints'],
                                     proxy_table['PSA_2024_Population'])[0])
    MEAN_ABS_DEV_PCT = float(proxy_table['Abs_Deviation_Pct'].mean())
    SIGNED_BIAS_PCT = float(proxy_table['Signed_Deviation_Pct'].mean())
    PROXY_VALIDATION_STATUS = 'COMPLETE'

    print('\n' + '=' * 92)
    print('  POPULATION PROXY MARGIN-OF-ERROR ASSESSMENT')
    print('=' * 92)
    print(proxy_table.to_string(index=False))
    print(f'\n   District population (PSA 2024) : {tot_pop:,}')
    print(f'   District footprints (OSM)      : {tot_prox:,}')
    print(f'   Implied persons per footprint  : {k:.2f}')
    print(f'   Pearson r                      : {PEARSON_R:.4f}')
    print(f'   Spearman ρ                     : {SPEARMAN_PROXY:.4f}')
    print(f'   Mean absolute deviation        : {MEAN_ABS_DEV_PCT:.2f}%')
    print(f'   Signed bias                    : {SIGNED_BIAS_PCT:+.2f}%')
    print('\n   Report all four separately. Carry the signed bias into Limitations.')
    proxy_table.to_csv(os.path.join(OUTPUT_DIR, 'population_proxy_validation.csv'),
                       index=False)

### Cell 17 — Tier 3 external checks

Two checks against independently observable conditions.

**Existing-facility concordance.** Each existing facility is matched to its nearest
candidate site and the distribution of suitability classes across those sites is
reported. Existing facilities are treated as a *revealed planning preference*, not as
ground truth: neither CHO nor CPDO has confirmed the present distribution as a normative
benchmark, and CHO consultation documented a facility rendered unsafe by inadequate
site-level hazard assessment. Note also that a facility's own location is by construction
a zero-coverage-gap point and is therefore penalised on that term, so the check tests
whether existing facilities sit in areas the framework rates plausibly on demand,
accessibility and hazard — not whether the framework would site a facility exactly where
one already stands.

**Population coverage improvement (F21).** The top 20 eligible candidate sites are
buffered at 2 km and 5 km and intersected with the building-footprint demand layer,
against the same buffers around the existing facility set. A positive ΔCoverage means the
framework's highest-ranked recommendations would extend population reach relative to the
current distribution. The buffers are Euclidean and inherit every limitation of the
footprint proxy; report them with that qualification attached.

In [ ]:
# =============================================================================
# CELL 17 — Tier 3 external checks
# Reports existing-facility concordance against the suitability classes, and the
# MARGINAL population coverage improvement (F21) of the top-20 eligible sites
# at 2 km and 5 km. Writes tier3_external_checks.json.

fac_xy_proj = np.column_stack([f_pts.x.values, f_pts.y.values])
d_near, i_near = cKDTree(site_xy).query(fac_xy_proj, k=1)
in_range = d_near <= (2 * GRID_SPACING_M)
near_class = df.iloc[i_near[in_range]]['suitability_class'].value_counts()

print('── Existing-facility concordance')
print(f'   Facilities matched to a candidate site within {2 * GRID_SPACING_M} m: '
      f'{int(in_range.sum())} of {len(fac_xy_proj)}')
CONCORDANCE = {}
for c in SUITABILITY_CLASSES:
    n = int(near_class.get(c, 0))
    CONCORDANCE[c] = n
    print(f'     {c:22s} {n:4d}  ({100 * n / max(int(in_range.sum()), 1):5.1f}%)')
CONCORDANCE_UPPER_PCT = float(100 * (CONCORDANCE[CLASS_HIGH] + CONCORDANCE[CLASS_MOD])
                              / max(int(in_range.sum()), 1))
print(f'   In the two upper tiers: {CONCORDANCE_UPPER_PCT:.1f}%')
print('   Read as directional plausibility against a revealed planning preference, '
      'never as confirmation that the framework identifies optimal sites.')

# PATCH — in-district facility set for the coverage comparison only.
# (CONCORDANCE above intentionally still uses the full matched set; this
# filter applies to the F21 coverage baseline specifically.)
fac_in_district_xy = fac_xy_proj[in_range]
N_FACILITIES_TOTAL = int(len(fac_xy_proj))
N_FACILITIES_IN_DISTRICT = int(in_range.sum())
print(f'\n   Coverage baseline uses the {N_FACILITIES_IN_DISTRICT} in-district facilities '
      f'only (of {N_FACILITIES_TOTAL} matched in the wider search buffer).')

print('\n── F21 Population coverage improvement (MARGINAL ΔCoverage)')
TOP_K_COVERAGE = 20
top_eligible = df[df['recommendation_eligible']].nlargest(TOP_K_COVERAGE, 'score')
top_xy = site_xy[top_eligible.index.values]

COVERAGE = {}
for buf in (2000, 5000):
    tree = cKDTree(b_xy)
    def cov_pct(pts):
        if len(pts) == 0:
            return 0.0
        hit = set()
        for p in pts:
            hit.update(tree.query_ball_point(p, r=buf))
        return 100.0 * len(hit) / len(b_xy)
    current_pct = cov_pct(fac_in_district_xy)
    combined_pct = cov_pct(np.vstack([fac_in_district_xy, top_xy]))
    marginal_pct = combined_pct - current_pct
    COVERAGE[buf] = {'current_pct': current_pct, 'combined_pct': combined_pct,
                     'marginal_gain_pct': marginal_pct}
    print(f'   {buf / 1000:.0f} km buffer — current (in-district): {current_pct:5.1f}%   '
          f'current + top-{TOP_K_COVERAGE}: {combined_pct:5.1f}%   '
          f'marginal gain: {marginal_pct:+.1f} pp')
print('   Marginal gain answers "how much additional demand would these sites reach '
      'beyond what existing facilities already cover" — the question Objective B '
      'actually poses. Buffers remain Euclidean and inherit the limitations of the '
      'footprint proxy; a low or negative marginal figure at large buffer radii is '
      'expected once existing coverage nears saturation, and should be read '
      'alongside the Tier 3 completeness findings and the documented Talomo North '
      'travel-time deficit, which a Euclidean buffer cannot represent.')

with open(os.path.join(OUTPUT_DIR, 'tier3_external_checks.json'), 'w') as f:
    json.dump({'facilities_total_matched': N_FACILITIES_TOTAL,
               'facilities_in_district': N_FACILITIES_IN_DISTRICT,
               'concordance_counts': CONCORDANCE,
               'concordance_upper_two_tiers_pct': CONCORDANCE_UPPER_PCT,
               'coverage_marginal': {str(k): v for k, v in COVERAGE.items()},
               'top_k_for_coverage': TOP_K_COVERAGE}, f, indent=2)

### Cell 18 — Figures and manuscript tables

Renders the interactive district map and writes every table the manuscript needs into a
single Excel workbook, `manuscript_tables.xlsx`, one sheet per table.

The map colours sites by suitability class, greys out setback-excluded sites, and draws
the district and barangay outlines, so the gate is visible rather than asserted. The
workbook collects the formula register, the barangay resolution table, per-barangay site
counts, feature summary statistics, model selection and evaluation, feature importance,
the Objective C comparison, class and hazard distributions, the ablation summary and,
where available, the population proxy table.

One workbook, one run. Anything pasted into the manuscript should come from this file
rather than from a scrollback buffer.

In [ ]:
# =============================================================================
# CELL 18 — Interactive map and consolidated manuscript tables
# Renders the Folium preview and writes manuscript_tables.xlsx, one sheet per
# table, so every figure pasted into the manuscript comes from a single run.
# =============================================================================
COLORS = {CLASS_HIGH: '#2e7d32', CLASS_MOD: '#1565c0',
          CLASS_LOW: '#f9a825', CLASS_NOT: '#c62828'}
m = folium.Map(location=[df['latitude'].mean(), df['longitude'].mean()],
               zoom_start=13, tiles='OpenStreetMap')
folium.GeoJson(gpd.GeoSeries([district_poly], crs=GEOGRAPHIC_CRS).__geo_interface__,
               name=STUDY_AREA,
               style_function=lambda x: {'color': '#111', 'weight': 2,
                                         'fillOpacity': 0}).add_to(m)
folium.GeoJson(brgy_gdf[['barangay', 'geometry']].__geo_interface__, name='Barangays',
               style_function=lambda x: {'color': '#666', 'weight': 1,
                                         'fillOpacity': 0}).add_to(m)
for _, r in df.iterrows():
    folium.CircleMarker(
        [r['latitude'], r['longitude']], radius=2.5,
        color=('#607d8b' if r['shoreline_excluded'] else COLORS[r['suitability_class']]),
        fill=True, fill_opacity=0.75,
        popup=folium.Popup(f"<b>{r['site_id']}</b><br>{r['barangay']}<br>"
                           f"Score {r['score']:.3f} — {r['suitability_class']}<br>"
                           f"{'ELIGIBLE' if r['recommendation_eligible'] else 'GATED'}",
                           max_width=260)).add_to(m)
folium.LayerControl().add_to(m)
m.save(os.path.join(OUTPUT_DIR, 'fig_candidate_site_map.html'))
print('fig_candidate_site_map.html saved.')
display(m)

# ── Consolidated workbook ────────────────────────────────────────────────────
sheets = {
    'Formula register': formula_register,
    'Barangay resolution': brgy_gdf[['barangay', 'osm_name', 'match_type', 'area_km2']],
    'Barangay site counts': pd.DataFrame(
        [{'Barangay': b, 'Candidate sites': BARANGAY_SITE_COUNTS[b]}
         for b in TALOMO_BARANGAYS]),
    'Feature summary': df[['building_density', 'road_distance', 'facility_distance',
                           'shoreline_distance']].describe().T.reset_index()
                        .rename(columns={'index': 'Feature'}),
    'Model selection': pd.read_csv(os.path.join(OUTPUT_DIR, 'model_selection.csv')),
    'Model evaluation': pd.DataFrame(
        [{'Metric': 'MAE (F10)', 'Value': MAE}, {'Metric': 'RMSE (F11)', 'Value': RMSE},
         {'Metric': 'R² (F12)', 'Value': R2}, {'Metric': 'OOB R² (F13)', 'Value': OOB_R2},
         {'Metric': 'CV R² mean', 'Value': CV_R2_MEAN},
         {'Metric': 'CV R² sd', 'Value': CV_R2_STD}]),
    'Feature importance': imp_table,
    'Objective C comparison': comparison,
    'Class distribution': pd.DataFrame(
        [{'Suitability class': c, 'Sites': CLASS_COUNTS[c],
          'Percent': round(100 * CLASS_COUNTS[c] / N_SITES, 2)}
         for c in SUITABILITY_CLASSES]),
    'Hazard distribution': pd.DataFrame(
        [{'Layer': h, **{HAZARD_LABELS[k]: HAZARD_DISTRIBUTIONS[h].get(k, 0)
                         for k in range(4)}} for h in HAZARD_FEATURES]),
    'Eligibility gates': pd.DataFrame(
        [{'Gate': f'Shoreline setback (<{SHORELINE_BUFFER_M} m)',
          'Sites barred': N_SHORELINE_EXCLUDED},
         {'Gate': 'High hazard on any layer', 'Sites barred': N_HIGH_HAZARD_EXCLUDED},
         {'Gate': 'Either gate (union)', 'Sites barred': N_INELIGIBLE},
         {'Gate': 'Eligible', 'Sites barred': N_ELIGIBLE}]),
    'Ablation': pd.DataFrame(
        [{'Measure': 'Spearman ρ (F16)', 'Value': SPEARMAN_RHO},
         {'Measure': 'Classification change % (F17)', 'Value': CLASSIFICATION_CHANGE_PCT},
         {'Measure': 'Mean deviation Δ̄ (F18)', 'Value': MEAN_DEVIATION}] +
        [{'Measure': f'Jaccard J({k}) (F19)', 'Value': v} for k, v in JACCARD.items()]),
    'Tier 3 coverage': pd.DataFrame(
        [{'Buffer (km)': k / 1000, **v} for k, v in COVERAGE.items()]),
    'Top recommendations': pd.DataFrame(top_recommendations),
}
if proxy_table is not None:
    sheets['Population proxy'] = proxy_table

xlsx_path = os.path.join(OUTPUT_DIR, 'manuscript_tables.xlsx')
with pd.ExcelWriter(xlsx_path, engine='xlsxwriter') as xw:
    for name, t in sheets.items():
        t.to_excel(xw, sheet_name=name[:31], index=False)
print(f'\nmanuscript_tables.xlsx written — {len(sheets)} sheets:')
for name in sheets:
    print(f'   {name}')

### Cell 19 — Manuscript synchronisation report

Prints every figure this run produced that also appears in the manuscript, in the order
the manuscript states them, with the section each belongs to. Saves the same text to
`manuscript_sync_report.txt`.

Use it as a checklist. Every number below replaces one currently printed in the
manuscript. The previous single-barangay run is superseded in full; do not mix figures
from the two runs in the same table, paragraph or figure caption.

The report ends with the items this run cannot settle — the ones that need the census
workbook, CHO or CPDO confirmation, or a decision recorded in writing.

In [ ]:
# =============================================================================
# CELL 19 — Manuscript synchronisation report
# Prints every figure from this run that appears in the manuscript, grouped by
# section, and lists the items this run cannot settle. Writes
# manuscript_sync_report.txt.
# =============================================================================
L = []
A = L.append
rule = lambda ch='=': A(ch * 78)

rule()
A('  POLARIS — MANUSCRIPT SYNCHRONISATION REPORT')
A(f'  Generated {datetime.now(timezone.utc).isoformat(timespec="seconds")}')
rule()
A('  Every figure below replaces one currently printed in the manuscript.')
A('  The previous single-barangay run is superseded IN FULL. Do not mix figures')
A('  from the two runs in the same table, paragraph or caption.')
A('')

rule('-')
A('  RESEARCH LOCALE / METHODOLOGY — study area and candidate grid')
rule('-')
A(f'  Barangays resolved                 : {len(brgy_gdf)} of {EXPECTED_BARANGAY_COUNT}')
A(f'  District area (from OSM polygons)  : {DISTRICT_AREA_KM2:.2f} km²')
A(f'  Grid spacing                       : {GRID_SPACING_M} m')
A(f'  Building-density radius            : {RADIUS_M} m')
A(f'  Candidate sites  n                 : {N_SITES:,}')
A(f'  Raw OSM elements read              : {RAW_OSM_RECORDS:,}')
A(f'  OSM snapshot date                  : {OSM_SNAPSHOT_DATE}')
A('')

rule('-')
A('  MATHEMATICAL FORMALIZATION — weights, thresholds, score ranges')
rule('-')
for f in ALL_FEATURES:
    sign = '+' if f in SERVICE_FEATURES else '−'
    A(f'  {f:28s} {sign}{WEIGHTS[f]:.2f}')
A(f'  Thresholds                         : τ₁={TAU1}, τ₂={TAU2}, τ₃={TAU3}')
A(f'  Raw hazard-integrated score range  : [{s_hz_raw.min():.4f}, {s_hz_raw.max():.4f}]')
A(f'  Raw service-only score range       : [{s_sv_raw.min():.4f}, {s_sv_raw.max():.4f}]')
A(f'  Predicted score range (exported)   : [{df.score.min():.3f}, {df.score.max():.3f}]')
A('')

rule('-')
A('  HARD ELIGIBILITY GATES')
rule('-')
A(f'  Shoreline setback (<{SHORELINE_BUFFER_M} m)          : {N_SHORELINE_EXCLUDED:,} sites')
A(f'  High hazard on any layer           : {N_HIGH_HAZARD_EXCLUDED:,} sites '
  f'({100 * N_HIGH_HAZARD_EXCLUDED / N_SITES:.1f}%)')
A(f'  Ineligible under either gate       : {N_INELIGIBLE:,} sites '
  f'({100 * N_INELIGIBLE / N_SITES:.1f}%)')
A(f'  Highest-scoring site overall       : {top_any.site_id} ({top_any.score:.3f}), '
  f'{"eligible" if top_any.recommendation_eligible else "GATED"}')
A(f'  Highest-scoring eligible site      : {top_elig.site_id} ({top_elig.score:.3f})')
A('')

rule('-')
A('  SUITABILITY CLASSIFICATION')
rule('-')
for c in SUITABILITY_CLASSES:
    A(f'  {c:22s} {CLASS_COUNTS[c]:6,d}  ({100 * CLASS_COUNTS[c] / N_SITES:5.1f}%)')
A('')

rule('-')
A('  TIER 2 — MODEL ACCURACY')
rule('-')
A(f'  Estimator                          : {SELECTED_MODEL_NAME}')
A(f'  Hyperparameters                    : {BEST_PARAMS}')
A(f'  MAE (F10)                          : {MAE:.4f}')
A(f'  RMSE (F11)                         : {RMSE:.4f}   (RMSE/MAE = {RMSE / MAE:.2f})')
A(f'  R² (F12)                           : {R2:.4f}')
A(f'  OOB R² (F13)                       : {OOB_R2:.4f}')
A(f'  5-fold CV R²                       : {CV_R2_MEAN:.4f} ± {CV_R2_STD:.4f}')
A(f'  Reconstruction |ŷ − S′| mean       : {dev.mean():.4f}; '
  f'{100 * DRIFT_OVER_005:.2f}% deviate > 0.05')
A('')
A('  Objective C — linear weighted overlay on the same partition:')
A(f'     OLS R²                          : {LINEAR_R2:.6f}')
A(f'     Random Forest R²                : {R2:.6f}')
A('     Report the Random Forest as a faithful surrogate, NOT as outperforming')
A('     the linear baseline. The target is linear by construction.')
A('')
A('  Feature importance (MDI / permutation ΔMSE / specified weight):')
for f in ALL_FEATURES:
    A(f'     {f:28s} {FEATURE_IMPORTANCES[f]:.4f} / '
      f'{PERMUTATION_IMPORTANCES[f]:.4f} / {WEIGHTS[f]:.2f}')
A('')

rule('-')
A('  TIER 2 — INTERNAL HAZARD SENSITIVITY ABLATION')
rule('-')
A(f'  Spearman ρ (F16)                   : {SPEARMAN_RHO:.4f}')
A(f'  Classification change C (F17)      : {CLASSIFICATION_CHANGE_PCT:.2f}% '
  f'({N_CHANGED:,} sites)')
A(f'  Mean deviation Δ̄ (F18, raw)        : {MEAN_DEVIATION:.4f}')
A('  Jaccard J(N) (F19)                 : ' +
  ', '.join(f'N={k}: {v:.3f}' for k, v in JACCARD.items()))
A('  Report ρ and C as describing the GRADUATED PENALTY ONLY. The gates are a')
A('  separate, larger mechanism and must be reported alongside, never merged.')
A('')

rule('-')
A('  TIER 3 — EXTERNAL VALIDITY')
rule('-')
A(f'  Population proxy validation        : {PROXY_VALIDATION_STATUS}')
if PROXY_VALIDATION_STATUS == 'COMPLETE':
    A(f'     Pearson r                       : {PEARSON_R:.4f}')
    A(f'     Spearman ρ                      : {SPEARMAN_PROXY:.4f}')
    A(f'     Mean absolute deviation         : {MEAN_ABS_DEV_PCT:.2f}%')
    A(f'     Signed bias                     : {SIGNED_BIAS_PCT:+.2f}%')
A(f'  Facility concordance, upper 2 tiers: {CONCORDANCE_UPPER_PCT:.1f}%')
for k, v in COVERAGE.items():
    A(f'  ΔCoverage at {k / 1000:.0f} km                 : {v["marginal_gain_pct"]:+.1f} pp '
      f'(POLARIS {v["combined_pct"]:.1f}% vs current {v["current_pct"]:.1f}%)')
A('')

rule('-')
A('  NOT SETTLED BY THIS RUN')
rule('-')
if PROXY_VALIDATION_STATUS != 'COMPLETE':
    A('  [ ] PSA 2024 POPCEN workbook — Cell 16 reports PENDING without it.')
A('  [ ] Threshold validation (τ₁, τ₂, τ₃) — expert confirmation, comparable')
A('      published scheme, or natural-breaks derivation. Panel: Ma\'am Regina.')
A('  [ ] CHO confirmation of the High-hazard gate, and whether a High class on a')
A('      5-year flood layer warrants the same treatment as one on a landslide layer.')
A('  [ ] CPDO review of the three service weights; CHO review of the three hazard')
A('      weights and their structural treatment.')
A('  [ ] Tiers 4 and 5 — expert validation and user acceptance evaluation.')
A('  [ ] Figure 3 (Conceptual Framework) — regenerate with the hard gates placed')
A('      after scoring.')
rule()

report = '\n'.join(L)
print(report)
with open(os.path.join(OUTPUT_DIR, 'manuscript_sync_report.txt'), 'w') as f:
    f.write(report)
print('\nmanuscript_sync_report.txt written.')